# US Power Outage Analysis

**Name(s)**: Layth Marabeh, Khanh Phan, Danny Xia   
**Repository Link**: https://github.com/k-phantastic/US-Power-Outage-Analysis   
**Website Link**: https://khanhphan.com/US-Power-Outage-Analysis

In [98]:
from pathlib import Path
import time

from utils import * 

pd.options.plotting.backend = 'plotly'
pio.renderers.default = "vscode" 
# For widescreen display, overrides utils.py settings
pd.set_option('display.max_columns', None)
pd.set_option("display.max_rows", None)
 
# Machine Learning Stack
from sklearn.model_selection import train_test_split, cross_val_score, RandomizedSearchCV
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler, QuantileTransformer
from sklearn.linear_model import LinearRegression, Ridge
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score, root_mean_squared_error
from sklearn.base import BaseEstimator, RegressorMixin
from sklearn.ensemble import RandomForestRegressor, RandomForestClassifier, HistGradientBoostingRegressor

from xgboost import XGBRegressor
import optuna
from scipy.stats import randint, uniform


## Step 1: Introduction

### U.S. Power Outage Dataset from Purdue University 

**Source:** https://engineering.purdue.edu/LASCI/research-data/outages/outage.xlsx  
**Data Dictionary:** https://www.sciencedirect.com/science/article/pii/S2352340918307182?via%3Dihub#t0005

>This dataset includes the major outages witnessed by different states in the continental U.S. Besides major outages, this data contains information on geographical location of the outages, regional climatic information, land-use characteristics, electricity consumption patterns and economic characteristics of the states affected by the outages. 

> Column information is located in Table 1 of Data Dictionary, Variable descriptions of the article

In [99]:
# Load the raw dataset
file_path = 'data/outage.xlsx'

raw_df = pd.read_excel(file_path)
raw_df.head(10)

,Major power outage events in the continental U.S.,Unnamed: 1,Unnamed: 2,Unnamed: 3,Unnamed: 4,Unnamed: 5,Unnamed: 6,Unnamed: 7,Unnamed: 8,Unnamed: 9,Unnamed: 10,Unnamed: 11,Unnamed: 12,Unnamed: 13,Unnamed: 14,Unnamed: 15,Unnamed: 16,Unnamed: 17,Unnamed: 18,Unnamed: 19,Unnamed: 20,Unnamed: 21,Unnamed: 22,Unnamed: 23,Unnamed: 24,Unnamed: 25,Unnamed: 26,Unnamed: 27,Unnamed: 28,Unnamed: 29,Unnamed: 30,Unnamed: 31,Unnamed: 32,Unnamed: 33,Unnamed: 34,Unnamed: 35,Unnamed: 36,Unnamed: 37,Unnamed: 38,Unnamed: 39,Unnamed: 40,Unnamed: 41,Unnamed: 42,Unnamed: 43,Unnamed: 44,Unnamed: 45,Unnamed: 46,Unnamed: 47,Unnamed: 48,Unnamed: 49,Unnamed: 50,Unnamed: 51,Unnamed: 52,Unnamed: 53,Unnamed: 54,Unnamed: 55,Unnamed: 56
0,Time period: January 2000 - July 2016,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,Regions affected: Outages reported in this dat...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,variables,OBS,YEAR,MONTH,U.S._STATE,POSTAL.CODE,NERC.REGION,CLIMATE.REGION,ANOMALY.LEVEL,CLIMATE.CATEGORY,OUTAGE.START.DATE,OUTAGE.START.TIME,OUTAGE.RESTORATION.DATE,OUTAGE.RESTORATION.TIME,CAUSE.CATEGORY,CAUSE.CATEGORY.DETAIL,HURRICANE.NAMES,OUTAGE.DURATION,DEMAND.LOSS.MW,CUSTOMERS.AFFECTED,RES.PRICE,COM.PRICE,IND.PRICE,TOTAL.PRICE,RES.SALES,COM.SALES,IND.SALES,TOTAL.SALES,RES.PERCEN,COM.PERCEN,IND.PERCEN,RES.CUSTOMERS,COM.CUSTOMERS,IND.CUSTOMERS,TOTAL.CUSTOMERS,RES.CUST.PCT,COM.CUST.PCT,IND.CUST.PCT,PC.REALGSP.STATE,PC.REALGSP.USA,PC.REALGSP.REL,PC.REALGSP.CHANGE,UTIL.REALGSP,TOTAL.REALGSP,UTIL.CONTRI,PI.UTIL.OFUSA,POPULATION,POPPCT_URBAN,POPPCT_UC,POPDEN_URBAN,POPDEN_UC,POPDEN_RURAL,AREAPCT_URBAN,AREAPCT_UC,PCT_LAND,PCT_WATER_TOT,PCT_WATER_INLAND
5,Units,NaN,NaN,NaN,NaN,NaN,NaN,NaN,numeric,NaN,"Day of the week, Month Day, Year",Hour:Minute:Second (AM / PM),"Day of the week, Month Day, Year",Hour:Minute:Second (AM / PM),NaN,NaN,NaN,mins,Megawatt,NaN,cents / kilowatt-hour,cents / kilowatt-hour,cents / kilowatt-hour,cents / kilowatt-hour,Megawatt-hour,Megawatt-hour,Megawatt-hour,Megawatt-hour,%,%,%,NaN,NaN,NaN,NaN,%,%,%,USD,USD,fraction,%,USD,USD,%,%,NaN,%,%,persons per square mile,persons per square mile,persons per square mile,%,%,%,%,%
6,NaN,1,2011,7,Minnesota,MN,MRO,East North Central,-0.3,normal,2011-07-01 00:00:00,17:00:00,2011-07-03 00:00:00,20:00:00,severe weather,NaN,NaN,3060,NaN,70000,11.6,9.18,6.81,9.28,2332915,2114774,2113291,6562520,35.55,32.23,32.2,2308736,276286,10673,2595696,88.94,10.64,0.41,51268,47586,1.08,1.6,4802,274182,1.75,2.2,5348119,73.27,15.28,2279,1700.5,18.2,2.14,0.6,91.59,8.41,5.48
7,NaN,2,2014,5,Minnesota,MN,MRO,East North Central,-0.1,normal,2014-05-11 00:00:00,18:38:00,2014-05-11 00:00:00,18:39:00,intentional attack,vandalism,NaN,1,NaN,NaN,12.12,9.71,6.49,9.28,1586986,1807756,1887927,5284231,30.03,34.21,35.73,2345860,284978,9898,2640737,88.83,10.79,0.37,53499,49091,1.09,1.9,5226,291955,1.79,2.2,5457125,73.27,15.28,2279,1700.5,18.2,2.14,0.6,91.59,8.41,5.48
8,NaN,3,2010,10,Minnesota,MN,MRO,East North Central,-1.5,cold,2010-10-26 00:00:00,20:00:00,2010-10-28 00:00:00,22:00:00,severe weather,heavy wind,NaN,3000,NaN,70000,10.87,8.19,6.07,8.15,1467293,1801683,1951295,5222116,28.1,34.5,37.37,2300291,276463,10150,2586905,

In [100]:
# Initial file has the header in row 5, with first column being blank and second column being index
df = pd.read_excel(file_path, header=5, usecols=range(2, 57), )

# Skip the units row
df = df.drop(index=0)
df.head()

,YEAR,MONTH,U.S._STATE,POSTAL.CODE,NERC.REGION,CLIMATE.REGION,ANOMALY.LEVEL,CLIMATE.CATEGORY,OUTAGE.START.DATE,OUTAGE.START.TIME,OUTAGE.RESTORATION.DATE,OUTAGE.RESTORATION.TIME,CAUSE.CATEGORY,CAUSE.CATEGORY.DETAIL,HURRICANE.NAMES,OUTAGE.DURATION,DEMAND.LOSS.MW,CUSTOMERS.AFFECTED,RES.PRICE,COM.PRICE,IND.PRICE,TOTAL.PRICE,RES.SALES,COM.SALES,IND.SALES,TOTAL.SALES,RES.PERCEN,COM.PERCEN,IND.PERCEN,RES.CUSTOMERS,COM.CUSTOMERS,IND.CUSTOMERS,TOTAL.CUSTOMERS,RES.CUST.PCT,COM.CUST.PCT,IND.CUST.PCT,PC.REALGSP.STATE,PC.REALGSP.USA,PC.REALGSP.REL,PC.REALGSP.CHANGE,UTIL.REALGSP,TOTAL.REALGSP,UTIL.CONTRI,PI.UTIL.OFUSA,POPULATION,POPPCT_URBAN,POPPCT_UC,POPDEN_URBAN,POPDEN_UC,POPDEN_RURAL,AREAPCT_URBAN,AREAPCT_UC,PCT_LAND,PCT_WATER_TOT,PCT_WATER_INLAND
1,2011.0,7.0,Minnesota,MN,MRO,East North Central,-0.3,normal,2011-07-01 00:00:00,17:00:00,2011-07-03 00:00:00,20:00:00,severe weather,NaN,NaN,3060,NaN,70000.0,11.6,9.18,6.81,9.28,2332915,2114774,2113291,6562520,35.55,32.23,32.2,2.31e+06,276286.0,10673.0,2.60e+06,88.94,10.64,0.41,51268,47586,1.08,1.6,4802,274182,1.75,2.2,5.35e+06,73.27,15.28,2279,1700.5,18.2,2.14,0.6,91.59,8.41,5.48
2,2014.0,5.0,Minnesota,MN,MRO,East North Central,-0.1,normal,2014-05-11 00:00:00,18:38:00,2014-05-11 00:00:00,18:39:00,intentional attack,vandalism,NaN,1,NaN,NaN,12.12,9.71,6.49,9.28,1586986,1807756,1887927,5284231,30.03,34.21,35.73,2.35e+06,284978.0,9898.0,2.64e+06,88.83,10.79,0.37,53499,49091,1.09,1.9,5226,291955,1.79,2.2,5.46e+06,73.27,15.28,2279,1700.5,18.2,2.14,0.6,91.59,8.41,5.48
3,2010.0,10.0,Minnesota,MN,MRO,East North Central,-1.5,cold,2010-10-26 00:00:00,20:00:00,2010-10-28 00:00:00,22:00:00,severe weather,heavy wind,NaN,3000,NaN,70000.0,10.87,8.19,6.07,8.15,1467293,1801683,1951295,5222116,28.1,34.5,37.37,2.30e+06,276463.0,10150.0,2.59e+06,88.92,10.69,0.39,50447,47287,1.07,2.7,4571,267895,1.71,2.1,5.31e+06,73.27,15.28,2279,1700.5,18.2,2.14,0.6,91.59,8.41,5.48
4,2012.0,6.0,Minnesota,MN,MRO,East North Central,-0.1,normal,2012-06-19 00:00:00,04:30:00,2012-06-20 00:00:00,23:00:00,severe weather,thunderstorm,NaN,2550,NaN,68200.0,11.79,9.25,6.71,9.19,1851519,1941174,1993026,5787064,31.99,33.54,34.44,2.32e+06,278466.0,11010.0,2.61e+06,88.9,10.68,0.42,51598,48156,1.07,0.6,5364,277627,1.93,2.2,5.38e+06,73.27,15.28,2279,1700.5,18.2,2.14,0.6,91.59,8.41,5.48
5,2015.0,7.0,Minnesota,MN,MRO,East North Central,1.2,warm,2015-07-18 00:00:00,02:00:00,2015-07-19 00:00:00,07:00:00,severe weather,NaN,NaN,1740,250,250000.0,13.07,10.16,7.74,10.43,2028875,2161612,1777937,5970339,33.98,36.21,29.78,2.37e+06,289044.0,9812.0,2.67e+06,88.82,10.81,0.37,54431,49844,1.09,1.7,4873,292023,1.67,2.2,5.49e+06,73.27,15.28,2279,1700.5,18.2,2.14,0.6,91.59,8.41,5.48


In [101]:
# DataFrame info
print(df.info())
print(f"\nColumns: {df.columns.tolist()}")
print(f"\nShape: {df.shape}")

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1534 entries, 1 to 1534
Data columns (total 55 columns):
 #   Column                   Non-Null Count  Dtype  
---  ------                   --------------  -----  
 0   YEAR                     1534 non-null   float64
 1   MONTH                    1525 non-null   float64
 2   U.S._STATE               1534 non-null   object 
 3   POSTAL.CODE              1534 non-null   object 
 4   NERC.REGION              1534 non-null   object 
 5   CLIMATE.REGION           1528 non-null   object 
 6   ANOMALY.LEVEL            1525 non-null   object 
 7   CLIMATE.CATEGORY         1525 non-null   object 
 8   OUTAGE.START.DATE        1525 non-null   object 
 9   OUTAGE.START.TIME        1525 non-null   object 
 10  OUTAGE.RESTORATION.DATE  1476 non-null   object 
 11  OUTAGE.RESTORATION.TIME  1476 non-null   object 
 12  CAUSE.CATEGORY           1534 non-null   object 
 13  CAUSE.CATEGORY.DETAIL    1063 non-null   object 
 14  HURRICANE.NAMES         

## Step 2: Data Cleaning and Exploratory Data Analysis

In [102]:
def fix_data_types(df):
    '''
    Fixes data types of columns in the DataFrame based on expected types.
    '''
    datetime_cols = [
        'OUTAGE.START.DATE', 
        #'OUTAGE.START.TIME',       # datetime.time object
        'OUTAGE.RESTORATION.DATE', 
        #'OUTAGE.RESTORATION.TIME'  # datetime.time object
        ]
    int_cols = [
        'YEAR', 'MONTH', 'OUTAGE.DURATION', 'DEMAND.LOSS.MW', 'CUSTOMERS.AFFECTED', 'POPULATION'
        ]
    float_cols = [
        'ANOMALY.LEVEL', 'RES.PRICE', 'COM.PRICE', 'IND.PRICE', 'TOTAL.PRICE', 'RES.SALES', 'COM.SALES', 'IND.SALES', 
        'TOTAL.SALES', 'RES.PERCEN', 'COM.PERCEN', 'IND.PERCEN', 'RES.CUSTOMERS', 'COM.CUSTOMERS', 'IND.CUSTOMERS', 
        'TOTAL.CUSTOMERS', 'RES.CUST.PCT', 'COM.CUST.PCT', 'IND.CUST.PCT', 'PC.REALGSP.STATE', 'PC.REALGSP.USA', 
        'PC.REALGSP.REL', 'PC.REALGSP.CHANGE', 'UTIL.REALGSP', 'TOTAL.REALGSP', 'UTIL.CONTRI', 'PI.UTIL.OFUSA',
        'POPPCT_URBAN', 'POPPCT_UC', 'POPDEN_URBAN', 'POPDEN_UC', 'POPDEN_RURAL', 'AREAPCT_URBAN', 'AREAPCT_UC', 'PCT_LAND', 
        'PCT_WATER_TOT', 'PCT_WATER_INLAND'
        ]
    # cat_cols = [
    #     'U.S._STATE', 'POSTAL.CODE', 'NERC.REGION', 'CLIMATE.REGION', 'CLIMATE.CATEGORY', 'CAUSE.CATEGORY', 
    #     'CAUSE.CATEGORY.DETAIL'
    # ]
    # HURRICANE.NAMES will be treated as object automatically 

    for col in datetime_cols:
        df[col] = pd.to_datetime(df[col], errors='coerce')
    for col in int_cols:
        df[col] = pd.to_numeric(df[col], errors='coerce').astype('Int64')
    for col in float_cols:
        df[col] = pd.to_numeric(df[col], errors='coerce').astype('float64')
    # for col in cat_cols:
    #     df[col] = df[col].astype('category')
    return df

In [103]:
# Columns with Missing Values
missing_values = df.isnull().sum()
print("\nMissing Values per Column:")
print(missing_values[missing_values > 0].sort_values(ascending=False))


Missing Values per Column:
HURRICANE.NAMES            1462
DEMAND.LOSS.MW              705
CAUSE.CATEGORY.DETAIL       471
CUSTOMERS.AFFECTED          443
OUTAGE.RESTORATION.DATE      58
OUTAGE.RESTORATION.TIME      58
OUTAGE.DURATION              58
RES.SALES                    22
TOTAL.PRICE                  22
IND.PRICE                    22
COM.PRICE                    22
TOTAL.SALES                  22
RES.PRICE                    22
IND.SALES                    22
RES.PERCEN                   22
COM.PERCEN                   22
IND.PERCEN                   22
COM.SALES                    22
POPDEN_UC                    10
POPDEN_RURAL                 10
OUTAGE.START.TIME             9
OUTAGE.START.DATE             9
CLIMATE.CATEGORY              9
ANOMALY.LEVEL                 9
MONTH                         9
CLIMATE.REGION                6
dtype: int64


In [104]:
# Missing CLIMATE.REGION values --> Hawaii and Alaska, non-continental states
missing_climate_region = df[df['CLIMATE.REGION'].isnull()]
missing_climate_region

,YEAR,MONTH,U.S._STATE,POSTAL.CODE,NERC.REGION,CLIMATE.REGION,ANOMALY.LEVEL,CLIMATE.CATEGORY,OUTAGE.START.DATE,OUTAGE.START.TIME,OUTAGE.RESTORATION.DATE,OUTAGE.RESTORATION.TIME,CAUSE.CATEGORY,CAUSE.CATEGORY.DETAIL,HURRICANE.NAMES,OUTAGE.DURATION,DEMAND.LOSS.MW,CUSTOMERS.AFFECTED,RES.PRICE,COM.PRICE,IND.PRICE,TOTAL.PRICE,RES.SALES,COM.SALES,IND.SALES,TOTAL.SALES,RES.PERCEN,COM.PERCEN,IND.PERCEN,RES.CUSTOMERS,COM.CUSTOMERS,IND.CUSTOMERS,TOTAL.CUSTOMERS,RES.CUST.PCT,COM.CUST.PCT,IND.CUST.PCT,PC.REALGSP.STATE,PC.REALGSP.USA,PC.REALGSP.REL,PC.REALGSP.CHANGE,UTIL.REALGSP,TOTAL.REALGSP,UTIL.CONTRI,PI.UTIL.OFUSA,POPULATION,POPPCT_URBAN,POPPCT_UC,POPDEN_URBAN,POPDEN_UC,POPDEN_RURAL,AREAPCT_URBAN,AREAPCT_UC,PCT_LAND,PCT_WATER_TOT,PCT_WATER_INLAND
1516,2008.0,12.0,Hawaii,HI,HI,NaN,-0.7,cold,2008-12-26 00:00:00,18:13:00,2008-12-27 00:00:00,17:00:00,severe weather,thunderstorm,NaN,1367,1060,294000.0,29.28,26.28,22.43,25.78,253625,275383,306397,835405,30.36,32.96,36.68,409668.0,61684.0,673.0,472025.0,86.79,13.07,0.14,50565,48401,1.04,-0.7,1646,67364,2.44,0.5,1.33e+06,91.93,20.47,3180.8,1664.7,18.2,6.12,2.6,58.75,41.25,0.38
1517,2011.0,5.0,Hawaii,HI,PR,NaN,-0.4,normal,2011-05-02 00:00:00,17:06:00,2011-05-02 00:00:00,20:00:00,severe weather,NaN,NaN,174,220,62000.0,34.58,32.14,27.85,31.29,249369,292304,310682,852355,29.26,34.29,36.45,417531.0,60043.0,698.0,478272.0,87.3,12.55,0.15,49110,47586,1.03,0.4,1935,67686,2.86,0.6,1.38e+06,91.93,20.47,3180.8,1664.7,18.2,6.12,2.6,58.75,41.25,0.38
1518,2006.0,10.0,Hawaii,HI,HECO,NaN,0.7,warm,2006-10-15 00:00:00,07:09:00,2006-10-15 00:00:00,16:12:00,severe weather,earthquake,NaN,543,110,59886.0,23.25,21.26,17.71,20.54,274609,307570,340735,922915,29.75,33.33,36.92,401592.0,61334.0,689.0,463615.0,86.62,13.23,0.15,50358,48909,1.03,1.1,1436,65956,2.18,0.5,1.31e+06,91.93,20.47,3180.8,1664.7,18.2,6.12,2.6,58.75,41.25,0.38
1519,2006.0,6.0,Hawaii,HI,HECO,NaN,0,normal,2006-06-01 00:00:00,14:12:00,2006-06-01 00:00:00,18:09:00,system operability disruption,NaN,NaN,237,120,29300.0,24.09,22.06,18.74,21.45,265474,298487,327618,891580,29.78,33.48,36.75,401592.0,61334.0,689.0,463615.0,86.62,13.23,0.15,50358,48909,1.03,1.1,1436,65956,2.18,0.5,1.31e+06,91.93,20.47,3180.8,1664.7,18.2,6.12,2.6,58.75,41.25,0.38
1520,2006.0,10.0,Hawaii,HI,HECO,NaN,0.7,warm,2006-10-15 00:00:00,07:09:00,2006-10-16 00:00:00,14:55:00,severe weather,earthquake,NaN,1906,1170,291000.0,23.25,21.26,17.71,20.54,274609,307570,340735,922915,29.75,33.33,36.92,401592.0,61334.0,689.0,463615.0,86.62,13.23,0.15,50358,48909,1.03,1.1,1436,65956,2.18,0.5,1.31e+06,91.93,20.47,3180.8,1664.7,18.2,6.12,2.6,58.75,41.25,0.38
1534,2000.0,NaN,Alaska,AK,ASCC,NaN,NaN,NaN,NaN,NaN,NaN,NaN,equipment failure,failure,NaN,NaN,35,14273.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,230534.0,38074.0,854.0,273530.0,84.28,13.92,0.31,57401,44745,1.28,-2.2,724,36046,2.01,0.2,6.28e+05,66.02,21.56,1802.6,1276,0.4,0.05,0.02,85.76,14.24,2.9


In [105]:
# Missing POPDEN_UC and POPDEN_RURAL values --> District of Columbia
missing_popden_uc_rural = df[df['POPDEN_UC'].isnull() | df['POPDEN_RURAL'].isnull()]
missing_popden_uc_rural

,YEAR,MONTH,U.S._STATE,POSTAL.CODE,NERC.REGION,CLIMATE.REGION,ANOMALY.LEVEL,CLIMATE.CATEGORY,OUTAGE.START.DATE,OUTAGE.START.TIME,OUTAGE.RESTORATION.DATE,OUTAGE.RESTORATION.TIME,CAUSE.CATEGORY,CAUSE.CATEGORY.DETAIL,HURRICANE.NAMES,OUTAGE.DURATION,DEMAND.LOSS.MW,CUSTOMERS.AFFECTED,RES.PRICE,COM.PRICE,IND.PRICE,TOTAL.PRICE,RES.SALES,COM.SALES,IND.SALES,TOTAL.SALES,RES.PERCEN,COM.PERCEN,IND.PERCEN,RES.CUSTOMERS,COM.CUSTOMERS,IND.CUSTOMERS,TOTAL.CUSTOMERS,RES.CUST.PCT,COM.CUST.PCT,IND.CUST.PCT,PC.REALGSP.STATE,PC.REALGSP.USA,PC.REALGSP.REL,PC.REALGSP.CHANGE,UTIL.REALGSP,TOTAL.REALGSP,UTIL.CONTRI,PI.UTIL.OFUSA,POPULATION,POPPCT_URBAN,POPPCT_UC,POPDEN_URBAN,POPDEN_UC,POPDEN_RURAL,AREAPCT_URBAN,AREAPCT_UC,PCT_LAND,PCT_WATER_TOT,PCT_WATER_INLAND
841,2010.0,2.0,District of Columbia,DC,RFC,Northeast,1.2,warm,2010-02-05 00:00:00,19:00:00,2010-02-12 00:00:00,15:46:00,severe weather,winter storm,NaN,9886,NaN,97651.0,13.41,13.8,7.11,13.55,181006,682695,18213,902386,20.06,75.65,2.02,227550.0,26449.0,1.0,254001.0,89.59,10.41,0.0,168377,47287,3.56,0.8,1416,101904,1.39,0.3,605126.0,100,0,9856.5,NaN,NaN,100,0,89.71,10.29,10.29
842,2011.0,8.0,District of Columbia,DC,RFC,Northeast,-0.6,cold,2011-08-27 00:00:00,23:05:00,2011-08-29 00:00:00,15:30:00,severe weather,NaN,NaN,2425,NaN,220000.0,13.06,12.77,8.29,12.69,239882,801888,16927,1084951,22.11,73.91,1.56,229450.0,26496.0,1.0,255948.0,89.65,10.35,0.0,167337,47586,3.52,-0.6,1300,103820,1.25,0.3,620472.0,100,0,9856.5,NaN,NaN,100,0,89.71,10.29,10.29
843,2010.0,8.0,District of Columbia,DC,RFC,Northeast,-1.2,cold,2010-08-05 00:00:00,15:30:00,2010-08-05 00:00:00,22:00:00,severe weather,thunderstorm,NaN,390,NaN,76729.0,14.76,13.31,8.13,13.47,236540,883096,18171,1167163,20.27,75.66,1.56,227550.0,26449.0,1.0,254001.0,89.59,10.41,0.0,168377,47287,3.56,0.8,1416,101904,1.39,0.3,605126.0,100,0,9856.5,NaN,NaN,100,0,89.71,10.29,10.29
844,2003.0,9.0,District of Columbia,DC,RFC,Northeast,0.2,normal,2003-09-18 00:00:00,16:20:00,2003-09-28 00:00:00,18:00:00,severe weather,hurricanes,Isabel,14500,NaN,530000.0,8.36,8.61,5.98,8.46,152325,721287,21491,920981,16.54,78.32,2.33,199215.0,26283.0,1.0,225500.0,88.34,11.66,0.0,155044,45858,3.38,3.3,1115,88143,1.26,0.4,568502.0,100,0,9856.5,NaN,NaN,100,0,89.71,10.29,10.29
845,2011.0,1.0,District of Columbia,DC,RFC,Northeast,-1.3,cold,2011-01-26 00:00:00,17:00:00,2011-01-31 00:00:00,08:00:00,severe weather,winter storm,NaN,6660,NaN,210000.0,13.62,13.17,7.48,13.13,229985,744163,19395,1018592,22.58,73.06,1.9,229450.0,26496.0,1.0,255948.0,89.65,10.35,0.0,167337,47586,3.52,-0.6,1300,103820,1.25,0.3,620472.0,100,0,9856.5,NaN,NaN,100,0,89.71,10.29,10.29
846,2013.0,2.0,District of Columbia,DC,RFC,Northeast,-0.4,normal,2013-02-08 00:00:00,11:38:00,2013-02-08 00:00:00,14:17:00,equipment failure,generator trip,NaN,159,140,52000.0,11.97,11.88,5.4,11.73,180750,627931,15521,846772,21.35,74.16,1.83,235322.0,26530.0,1.0,261854.0,89.87,10.13,0.0,159340,48396,3.29,-2.5,1165,103430,1.13,0.3,649540.0,100,0,9856.5,NaN,NaN,100,0,89.71,10.29,10.29
847,2014.0,1.0,District of Columbia,DC,RFC,Northeast,-0.5,cold,2014-01-06 00:00:00,19:50:00,2014-01-06 00:00:00,20:44:00,severe weather,winter,NaN,54,NaN,NaN,12.59,12.67,8.66,12.49,218682,736898,20478,1004907,21.76,73.33,2.04,239355.0,26463.0,1.0,265820.0,90.04,9.96,0.0,159831,49091,3.26,0.3,1107,105312,1.05,0.3,659836.0,100,0,9856.5,NaN,NaN,100,0,89.71,10.29,10.29
848,2013.0,6.0,District of Columbia,DC,RFC,Northeast,-0.2,normal,2013-06-13 00:00:00,15:30:00,2013-06-13 00:00:00,16:00:00,severe weather,uncontrolled loss,NaN,30,700,40000.0,12.65,11.77,5.33,11.72,175324,789494,20375,1015794,17.26,77.72,2.01,235322.0,26530.0,1.0,261854.0,89.87,10.13,0.0,159340,48396,3.29,-2.5,1165,103430,1.13,0.3,649540.0,100,0,9856.5,NaN,NaN,100,0,89.71,10.29,10.29
849,2012.0,6.0,District of Columbia,DC,RFC,Northeast,-0.1,normal,2012-06-29 00:00:00,22:15:00,2012-07-05 00:00:00,12:52:00,severe weather,thunderstorm,NaN,8077,3000,425000.0,13.18,12.03,5.71,12.03,179048,772663,18518,9

In [106]:
# Show duplicates
df_duplicates = df[df.duplicated(keep=False)]
df_duplicates

,YEAR,MONTH,U.S._STATE,POSTAL.CODE,NERC.REGION,CLIMATE.REGION,ANOMALY.LEVEL,CLIMATE.CATEGORY,OUTAGE.START.DATE,OUTAGE.START.TIME,OUTAGE.RESTORATION.DATE,OUTAGE.RESTORATION.TIME,CAUSE.CATEGORY,CAUSE.CATEGORY.DETAIL,HURRICANE.NAMES,OUTAGE.DURATION,DEMAND.LOSS.MW,CUSTOMERS.AFFECTED,RES.PRICE,COM.PRICE,IND.PRICE,TOTAL.PRICE,RES.SALES,COM.SALES,IND.SALES,TOTAL.SALES,RES.PERCEN,COM.PERCEN,IND.PERCEN,RES.CUSTOMERS,COM.CUSTOMERS,IND.CUSTOMERS,TOTAL.CUSTOMERS,RES.CUST.PCT,COM.CUST.PCT,IND.CUST.PCT,PC.REALGSP.STATE,PC.REALGSP.USA,PC.REALGSP.REL,PC.REALGSP.CHANGE,UTIL.REALGSP,TOTAL.REALGSP,UTIL.CONTRI,PI.UTIL.OFUSA,POPULATION,POPPCT_URBAN,POPPCT_UC,POPDEN_URBAN,POPDEN_UC,POPDEN_RURAL,AREAPCT_URBAN,AREAPCT_UC,PCT_LAND,PCT_WATER_TOT,PCT_WATER_INLAND
25,2014.0,1.0,Tennessee,TN,SERC,Central,-0.5,cold,2014-01-07 00:00:00,06:00:00,2014-01-07 00:00:00,08:30:00,severe weather,winter,NaN,150,NaN,NaN,9.73,10.02,6.27,9.13,4795011,2841462,1912538,9549176,50.21,29.76,20.03,2.76e+06,4.72e+05,1271.0,3.23e+06,85.35,14.61,0.04,41955,49091,0.85,0.8,1739,274775,0.63,0.4,6.55e+06,66.39,12.02,1450.3,1076.2,55.6,7.05,1.72,97.84,2.16,2.16
44,2014.0,1.0,Tennessee,TN,SERC,Central,-0.5,cold,2014-01-07 00:00:00,06:00:00,2014-01-07 00:00:00,08:30:00,severe weather,winter,NaN,150,NaN,NaN,9.73,10.02,6.27,9.13,4795011,2841462,1912538,9549176,50.21,29.76,20.03,2.76e+06,4.72e+05,1271.0,3.23e+06,85.35,14.61,0.04,41955,49091,0.85,0.8,1739,274775,0.63,0.4,6.55e+06,66.39,12.02,1450.3,1076.2,55.6,7.05,1.72,97.84,2.16,2.16
220,2013.0,12.0,Texas,TX,TRE,South,-0.3,normal,2013-12-13 00:00:00,11:00:00,2013-12-27 00:00:00,11:00:00,fuel supply emergency,Coal,NaN,20160,NaN,NaN,11.27,8,5.79,8.67,11901122,10736864,8269474,30908046,38.5,34.74,26.76,9.95e+06,1.37e+06,95881.0,1.14e+07,87.19,11.97,0.84,51695,48396,1.07,2.7,28880,1370216,2.11,10.5,2.65e+07,84.7,9.35,2435.3,1539.9,15.2,3.35,0.58,97.26,2.74,2.09
274,2013.0,12.0,Texas,TX,TRE,South,-0.3,normal,2013-12-13 00:00:00,11:00:00,2013-12-27 00:00:00,11:00:00,fuel supply emergency,Coal,NaN,20160,NaN,NaN,11.27,8,5.79,8.67,11901122,10736864,8269474,30908046,38.5,34.74,26.76,9.95e+06,1.37e+06,95881.0,1.14e+07,87.19,11.97,0.84,51695,48396,1.07,2.7,28880,1370216,2.11,10.5,2.65e+07,84.7,9.35,2435.3,1539.9,15.2,3.35,0.58,97.26,2.74,2.09
500,2004.0,7.0,Arizona,AZ,WECC,Southwest,0.5,warm,2004-07-06 00:00:00,06:00:00,2004-08-09 00:00:00,12:00:00,severe weather,wildfire,NaN,49320,NaN,NaN,8.88,7.57,5.55,7.91,3466457,2563737,1048662,7078856,48.97,36.22,14.81,2.26e+06,2.59e+05,7419.0,2.52e+06,89.44,10.26,0.29,41169,47037,0.88,1,5113,232702,2.2,1.7,5.65e+06,89.81,9.74,2625.4,1669,5.8,1.92,0.33,99.65,0.35,0.35
514,2004.0,7.0,Arizona,AZ,WECC,Southwest,0.5,warm,2004-07-06 00:00:00,06:00:00,2004-08-09 00:00:00,12:00:00,severe weather,wildfire,NaN,49320,NaN,NaN,8.88,7.57,5.55,7.91,3466457,2563737,1048662,7078856,48.97,36.22,14.81,2.26e+06,2.59e+05,7419.0,2.52e+06,89.44,10.26,0.29,41169,47037,0.88,1,5113,232702,2.2,1.7,5.65e+06,89.81,9.74,2625.4,1669,5.8,1.92,0.33,99.65,0.35,0.35
994,2010.0,8.0,Louisiana,LA,SERC,South,-1.2,cold,2010-08-02 00:00:00,12:45:00,2010-08-04 00:00:00,11:00:00,public appeal,NaN,NaN,2775,NaN,NaN,9.29,8.5,5.99,8.13,3695516,2407508,2401355,8505369,43.45,28.31,28.23,1.97e+06,2.74e+05,18128.0,2.27e+06,87.1,12.1,0.8,48305,47287,1.02,3.6,4509,219576,2.05,1.4,4.54e+06,73.19,11.85,1685.8,1280.1,29.5,4.56,0.97,82.49,17.51,8.71
998,2010.0,8.0,Louisiana,LA,SERC,South,-1.2,cold,2010-08-02 00:00:00,12:45:00,2010-08-04 00:00:00,11:00:00,public appeal,NaN,NaN,2775,NaN,NaN,9.29,8.5,5.99,8.13,3695516,2407508,2401355,8505369,43.45,28.31,28.23,1.97e+06,2.74e+05,18128.0,2.27e+06,87.1,12.1,0.8,48305,47287,1.02,3.6,4509,219576,2.05,1.4,4.54e+06,73.19,11.85,1685.8,1280.1,29.5,4.56,0.97,82.49,17.51,8.71
1000,2010.0,8.0,Louisiana,LA,SERC,South,-1.2,cold,2010-08-02 00:00:00,12:45:00,2010-08-04 00:00:00,11:00:00,public appeal,NaN,NaN,2775,NaN,NaN,9.29,8.5,5.99,8.13,3695516,2407508,2401355,8505369,43.45,28.31,28.23,1.97e+06,2.74e+05,18128.0,2.27e+06,87.1,12.1,

In [107]:
df[df['OUTAGE.DURATION'] > 30000]

,YEAR,MONTH,U.S._STATE,POSTAL.CODE,NERC.REGION,CLIMATE.REGION,ANOMALY.LEVEL,CLIMATE.CATEGORY,OUTAGE.START.DATE,OUTAGE.START.TIME,OUTAGE.RESTORATION.DATE,OUTAGE.RESTORATION.TIME,CAUSE.CATEGORY,CAUSE.CATEGORY.DETAIL,HURRICANE.NAMES,OUTAGE.DURATION,DEMAND.LOSS.MW,CUSTOMERS.AFFECTED,RES.PRICE,COM.PRICE,IND.PRICE,TOTAL.PRICE,RES.SALES,COM.SALES,IND.SALES,TOTAL.SALES,RES.PERCEN,COM.PERCEN,IND.PERCEN,RES.CUSTOMERS,COM.CUSTOMERS,IND.CUSTOMERS,TOTAL.CUSTOMERS,RES.CUST.PCT,COM.CUST.PCT,IND.CUST.PCT,PC.REALGSP.STATE,PC.REALGSP.USA,PC.REALGSP.REL,PC.REALGSP.CHANGE,UTIL.REALGSP,TOTAL.REALGSP,UTIL.CONTRI,PI.UTIL.OFUSA,POPULATION,POPPCT_URBAN,POPPCT_UC,POPDEN_URBAN,POPDEN_UC,POPDEN_RURAL,AREAPCT_URBAN,AREAPCT_UC,PCT_LAND,PCT_WATER_TOT,PCT_WATER_INLAND
54,2014.0,1.0,Wisconsin,WI,RFC,East North Central,-0.5,cold,2014-01-24 00:00:00,00:00:00,2014-04-09 00:00:00,11:53:00,fuel supply emergency,Coal,NaN,108653,NaN,NaN,12.86,10.24,7.16,10.28,2393637,2089309,1959985,6442931,37.15,32.43,30.42,2.63e+06,3.46e+05,5465.0,2.98e+06,88.22,11.6,0.18,46676,49091,0.95,1.9,4680,268742,1.74,1.9,5.76e+06,70.15,14.35,2123.3,1671.5,32.5,3.47,0.9,82.69,17.31,3.05
118,2003.0,5.0,Michigan,MI,RFC,East North Central,-0.2,normal,2003-05-15 00:00:00,14:00:00,2003-06-16 00:00:00,14:00:00,severe weather,flooding,NaN,46080,240,2.0,8.23,7.45,4.95,6.67,2244807,2789455,3273564,8308074,27.02,33.58,39.4,4.22e+06,4.83e+05,14224.0,4.71e+06,89.45,10.25,0.3,42768,45858,0.93,1.7,8288,429436,1.93,3.8,1.00e+07,74.57,8.19,2034.1,1390.4,47.5,6.41,1.03,58.46,41.54,2.07
121,2009.0,3.0,Michigan,MI,RFC,East North Central,-0.4,normal,2009-03-03 00:00:00,06:48:00,2009-04-26 00:00:00,18:05:00,equipment failure,transformer outage,NaN,78377,378,NaN,10.66,8.78,6.63,8.84,2825441,3019167,2251830,8096999,34.89,37.29,27.81,4.25e+06,5.21e+05,13065.0,4.79e+06,88.85,10.87,0.27,37004,46680,0.79,-7.7,7769,366401,2.12,3.7,9.90e+06,74.57,8.19,2034.1,1390.4,47.5,6.41,1.03,58.46,41.54,2.07
500,2004.0,7.0,Arizona,AZ,WECC,Southwest,0.5,warm,2004-07-06 00:00:00,06:00:00,2004-08-09 00:00:00,12:00:00,severe weather,wildfire,NaN,49320,NaN,NaN,8.88,7.57,5.55,7.91,3466457,2563737,1048662,7078856,48.97,36.22,14.81,2.26e+06,2.59e+05,7419.0,2.52e+06,89.44,10.26,0.29,41169,47037,0.88,1,5113,232702,2.2,1.7,5.65e+06,89.81,9.74,2625.4,1669,5.8,1.92,0.33,99.65,0.35,0.35
514,2004.0,7.0,Arizona,AZ,WECC,Southwest,0.5,warm,2004-07-06 00:00:00,06:00:00,2004-08-09 00:00:00,12:00:00,severe weather,wildfire,NaN,49320,NaN,NaN,8.88,7.57,5.55,7.91,3466457,2563737,1048662,7078856,48.97,36.22,14.81,2.26e+06,2.59e+05,7419.0,2.52e+06,89.44,10.26,0.29,41169,47037,0.88,1,5113,232702,2.2,1.7,5.65e+06,89.81,9.74,2625.4,1669,5.8,1.92,0.33,99.65,0.35,0.35
990,2014.0,2.0,New York,NY,NPCC,Northeast,-0.5,cold,2014-02-07 00:00:00,07:00:00,2014-03-21 00:00:00,08:00:00,fuel supply emergency,Coal,NaN,60480,675,NaN,21.69,17.48,8.19,17.81,4550076,6390265,1497619,12699456,35.83,50.32,11.79,7.05e+06,1.05e+06,7862.0,8.10e+06,87.0,12.91,0.1,63217,49091,1.29,1,18048,1248299,1.45,7.3,1.97e+07,87.87,5.21,4161.4,1700,54.6,8.68,1.26,86.38,13.62,3.65
1182,2003.0,10.0,California,CA,WECC,West,0.3,normal,2003-10-26 00:00:00,01:44:00,2003-11-18 00:00:00,22:54:00,severe weather,wildfire,NaN,34390,NaN,108000.0,10.47,11.64,9.17,10.72,7096893,10217148,4590049,21974546,32.3,46.5,20.89,1.21e+07,1.77e+06,82579.0,1.40e+07,86.74,12.66,0.59,49815,45858,1.09,2.9,28280,1756127,1.61,10.8,3.53e+07,94.95,5.22,4303.7,2124.1,12.7,5.28,0.59,95.16,4.84,1.73
1208,2016.0,3.0,California,CA,WECC,West,1.6,warm,2016-03-03 00:00:00,11:00:00,2016-04-06 00:00:00,19:47:00,fuel supply emergency,NaN,NaN,49427,0,0.0,17.66,13.83,10.39,14.34,6448036,9366373,4149987,20033300,32.19,46.75,20.72,1.34e+07,1.69e+06,148549.0,1.53e+07,87.96,11.07,0.97,58974,50660,1.16,4.6,27735,2317466,1.2,12,3.93e+07,94.95,5.22,4303.7,2124.1,12.7,5.28,0.59,95.16,4.84,1.73


In [108]:
# Data Cleaning Functions
def remove_duplicate_rows(df):
    # Remove duplicate rows 
    df = df.drop_duplicates()
    return df

def combine_outage_start(df): 
    #Combine OUTAGE.START.DATE and OUTAGE.START.TIME into a single column OUTAGE.START.
    time_as_td = pd.to_timedelta(df['OUTAGE.START.TIME'].astype(str), errors='coerce')
    df['OUTAGE.START'] = df['OUTAGE.START.DATE'] + time_as_td
    return df

def combine_outage_restoration(df): 
    # Combine OUTAGE.RESTORATION.DATE and OUTAGE.RESTORATION.TIME into a single column OUTAGE.RESTORATION.
    time_as_td = pd.to_timedelta(df['OUTAGE.RESTORATION.TIME'].astype(str), errors='coerce')
    df['OUTAGE.RESTORATION'] = df['OUTAGE.RESTORATION.DATE'] + time_as_td
    return df

def add_month_names(df):
    # Map numeric MONTH to MONTH.NAME.
    MONTH_NAMES = {1:'Jan',2:'Feb',3:'Mar',4:'Apr',5:'May',6:'Jun',
                   7:'Jul',8:'Aug',9:'Sep',10:'Oct',11:'Nov',12:'Dec'}
    df['MONTH.NAME'] = df['MONTH'].map(MONTH_NAMES)
    return df

In [109]:
# Apply all cleaning functions in a pipeline

df_cleaned = (
    df.pipe(remove_duplicate_rows)
      .pipe(fix_data_types)
      .pipe(combine_outage_start)
      .pipe(combine_outage_restoration)
      .pipe(add_month_names)
)

In [110]:
df_cleaned.head()

,YEAR,MONTH,U.S._STATE,POSTAL.CODE,NERC.REGION,CLIMATE.REGION,ANOMALY.LEVEL,CLIMATE.CATEGORY,OUTAGE.START.DATE,OUTAGE.START.TIME,OUTAGE.RESTORATION.DATE,OUTAGE.RESTORATION.TIME,CAUSE.CATEGORY,CAUSE.CATEGORY.DETAIL,HURRICANE.NAMES,OUTAGE.DURATION,DEMAND.LOSS.MW,CUSTOMERS.AFFECTED,RES.PRICE,COM.PRICE,IND.PRICE,TOTAL.PRICE,RES.SALES,COM.SALES,IND.SALES,TOTAL.SALES,RES.PERCEN,COM.PERCEN,IND.PERCEN,RES.CUSTOMERS,COM.CUSTOMERS,IND.CUSTOMERS,TOTAL.CUSTOMERS,RES.CUST.PCT,COM.CUST.PCT,IND.CUST.PCT,PC.REALGSP.STATE,PC.REALGSP.USA,PC.REALGSP.REL,PC.REALGSP.CHANGE,UTIL.REALGSP,TOTAL.REALGSP,UTIL.CONTRI,PI.UTIL.OFUSA,POPULATION,POPPCT_URBAN,POPPCT_UC,POPDEN_URBAN,POPDEN_UC,POPDEN_RURAL,AREAPCT_URBAN,AREAPCT_UC,PCT_LAND,PCT_WATER_TOT,PCT_WATER_INLAND,OUTAGE.START,OUTAGE.RESTORATION,MONTH.NAME
1,2011,7,Minnesota,MN,MRO,East North Central,-0.3,normal,2011-07-01,17:00:00,2011-07-03,20:00:00,severe weather,NaN,NaN,3060,<NA>,70000,11.60,9.18,6.81,9.28,2.33e+06,2.11e+06,2.11e+06,6.56e+06,35.55,32.23,32.20,2.31e+06,276286.0,10673.0,2.60e+06,88.94,10.64,0.41,51268.0,47586.0,1.08,1.6,4802.0,274182.0,1.75,2.2,5348119,73.27,15.28,2279.0,1700.5,18.2,2.14,0.6,91.59,8.41,5.48,2011-07-01 17:00:00,2011-07-03 20:00:00,Jul
2,2014,5,Minnesota,MN,MRO,East North Central,-0.1,normal,2014-05-11,18:38:00,2014-05-11,18:39:00,intentional attack,vandalism,NaN,1,<NA>,<NA>,12.12,9.71,6.49,9.28,1.59e+06,1.81e+06,1.89e+06,5.28e+06,30.03,34.21,35.73,2.35e+06,284978.0,9898.0,2.64e+06,88.83,10.79,0.37,53499.0,49091.0,1.09,1.9,5226.0,291955.0,1.79,2.2,5457125,73.27,15.28,2279.0,1700.5,18.2,2.14,0.6,91.59,8.41,5.48,2014-05-11 18:38:00,2014-05-11 18:39:00,May
3,2010,10,Minnesota,MN,MRO,East North Central,-1.5,cold,2010-10-26,20:00:00,2010-10-28,22:00:00,severe weather,heavy wind,NaN,3000,<NA>,70000,10.87,8.19,6.07,8.15,1.47e+06,1.80e+06,1.95e+06,5.22e+06,28.10,34.50,37.37,2.30e+06,276463.0,10150.0,2.59e+06,88.92,10.69,0.39,50447.0,47287.0,1.07,2.7,4571.0,267895.0,1.71,2.1,5310903,73.27,15.28,2279.0,1700.5,18.2,2.14,0.6,91.59,8.41,5.48,2010-10-26 20:00:00,2010-10-28 22:00:00,Oct
4,2012,6,Minnesota,MN,MRO,East North Central,-0.1,normal,2012-06-19,04:30:00,2012-06-20,23:00:00,severe weather,thunderstorm,NaN,2550,<NA>,68200,11.79,9.25,6.71,9.19,1.85e+06,1.94e+06,1.99e+06,5.79e+06,31.99,33.54,34.44,2.32e+06,278466.0,11010.0,2.61e+06,88.90,10.68,0.42,51598.0,48156.0,1.07,0.6,5364.0,277627.0,1.93,2.2,5380443,73.27,15.28,2279.0,1700.5,18.2,2.14,0.6,91.59,8.41,5.48,2012-06-19 04:30:00,2012-06-20 23:00:00,Jun
5,2015,7,Minnesota,MN,MRO,East North Central,1.2,warm,2015-07-18,02:00:00,2015-07-19,07:00:00,severe weather,NaN,NaN,1740,250,250000,13.07,10.16,7.74,10.43,2.03e+06,2.16e+06,1.78e+06,5.97e+06,33.98,36.21,29.78,2.37e+06,289044.0,9812.0,2.67e+06,88.82,10.81,0.37,54431.0,49844.0,1.09,1.7,4873.0,292023.0,1.67,2.2,5489594,73.27,15.28,2279.0,1700.5,18.2,2.14,0.6,91.59,8.41,5.48,2015-07-18 02:00:00,2015-07-19 07:00:00,Jul


In [111]:
# Export cleaned dataset to CSV
output_path = 'data/cleaned_outage_dataset.csv'
df_cleaned.to_csv(output_path, index=False)
print(f"Cleaned dataset exported to {output_path}")

Cleaned dataset exported to data/cleaned_outage_dataset.csv


# Exploratory Data Analysis

In [112]:
# Outage Frequency over the Years
yearly = df_cleaned.groupby('YEAR', observed=True).size().reset_index(name='Number of Outages')
fig = px.scatter(
    yearly, 
    x='YEAR', 
    y='Number of Outages', 
    title='<b>Number of Outages per Year</b>',
    labels={'Number of Outages': 'Number of Outages', 'YEAR': 'Year'},
    trendline='ols' 
)

fig.add_trace(go.Scatter(
    x=yearly['YEAR'], 
    y=yearly['Number of Outages'], 
    mode='lines', 
    name='Trend', 
    line=dict(color='teal', width=1),
    showlegend=False
))
fig.data[1].line.color = 'red'
fig.data[1].line.dash = 'dot'
fig.update_layout( 
    template="plotly_white" # Match style of other plots
)
fig.show()

# Export plot for website
fig.write_html('docs/assets/plots/outages_per_year_all.html', include_plotlyjs='cdn')

In [113]:
yearly_by_region = df_cleaned.groupby(['YEAR', 'CLIMATE.REGION'], observed=True).size().unstack(fill_value=0)
fig = px.line(
    yearly_by_region.reset_index(), 
    x='YEAR', 
    y=yearly_by_region.columns.tolist(), 
    title='<b>Number of Outages by Year and Region</b>',
    labels={'value': 'Number of Outages', 'variable': 'Region', 'YEAR': 'Year'},
)
fig.update_layout(legend_title_text='NCEI Climate Region')
fig.update_layout( 
    template="plotly_white" # Match style of other plots
)
fig.show()

# Export plot for website
fig.write_html('docs/assets/plots/outages_per_year_by_region.html', include_plotlyjs='cdn')

In [114]:
# Outage Frequency by Month
month_order = ['Jan','Feb','Mar','Apr','May','Jun','Jul','Aug','Sep','Oct','Nov','Dec']

by_month = df_cleaned.groupby('MONTH.NAME', observed=True).size().reindex(month_order)
fig = px.bar(
    by_month.reset_index(name='Number of Outages'), 
    x='MONTH.NAME', 
    y='Number of Outages', 
    title='<b>Number of Outages by Month (all years combined)</b>',
    labels={'MONTH.NAME': 'Month', 'Number of Outages': 'Number of Outages'},
    color='Number of Outages',
    color_continuous_scale='teal',
)
fig.update_layout(legend_title_text='Count')
fig.update_layout( 
    template="plotly_white" # Match style of other plots
)
fig.show()

# Export plot for website
fig.write_html('docs/assets/plots/outages_by_month.html', include_plotlyjs='cdn')

In [115]:
by_region = df_cleaned.groupby('CLIMATE.REGION', observed=True).size().sort_values(ascending=False)
fig = px.bar(
    by_region.reset_index(name='Number of Outages'), 
    x='CLIMATE.REGION', 
    y='Number of Outages', 
    title='<b>Total Number of Outages by Climate Region</b>',
    labels={'CLIMATE.REGION': 'NCEI Climate Region', 'Number of Outages': 'Number of Outages'},
    color='Number of Outages',
    color_continuous_scale='teal',
)
fig.update_layout( 
    template="plotly_white" # Match style of other plots
)
fig.show()

# Export plot for website
fig.write_html('docs/assets/plots/outages_by_climate_region.html', include_plotlyjs='cdn')

In [116]:
# Regional Analysis
top_15_states = df_cleaned['U.S._STATE'].value_counts().nlargest(15).sort_values(ascending=True)
fig = px.bar(
    top_15_states.reset_index(name='Number of Outages'), 
    x='Number of Outages', 
    y='U.S._STATE', 
    title='<b>Top 15 States by Number of Outages</b>',
    labels={'U.S._STATE': 'State', 'Number of Outages': 'Number of Outages'},
    color='Number of Outages',
    color_continuous_scale='teal',
    orientation='h'
)
fig.update_layout( 
    template="plotly_white" # Match style of other plots
)
fig.show()

# Export plot for website
fig.write_html('docs/assets/plots/top_15_states_by_outages.html', include_plotlyjs='cdn')

In [117]:
by_cause = (
    df_cleaned
    .groupby('CAUSE.CATEGORY', observed=True)
    .agg(
        num_outages=('OUTAGE.DURATION', 'count'),
        avg_duration=('OUTAGE.DURATION', 'mean'),
    )
    .sort_values('num_outages')
    .reset_index()
)

fig = px.bar(
    by_cause,
    x='num_outages',
    y='CAUSE.CATEGORY',
    orientation='h',
    color='avg_duration',                      
    color_continuous_scale='teal',
    title='<b>Number of Outages by Cause Category</b><br>Colored by Average Outage Duration (in minutes)',
    labels={
        'num_outages':     'Number of Outages',
        'CAUSE.CATEGORY':  'Cause Category',
        'avg_duration':    'Avg Outage Duration (min)',
    },
    hover_data={'avg_duration': ':.0f'},
)
fig.update_layout(coloraxis_colorbar_title='Avg Duration<br>(min)')
fig.update_layout( 
    template="plotly_white" # Match style of other plots
)

fig.show()

# Export plot for website
fig.write_html('docs/assets/plots/outages_by_cause_category.html', include_plotlyjs='cdn')

In [118]:
# Relevant climate region map: https://www.ncei.noaa.gov/access/monitoring/reference-maps/us-climate-regions
pivot_df = (
    df_cleaned
    .groupby(['CAUSE.CATEGORY', 'CLIMATE.REGION'], observed=True)
    .agg(avg_outage_duration = ('OUTAGE.DURATION', 'mean'))
    .reset_index()
)

pivot_table = pivot_df.pivot_table(
    index='CAUSE.CATEGORY',
    columns='CLIMATE.REGION',
    values='avg_outage_duration',
)

region_abbrev = {
    'East North Central': 'E. N. Central',
    'West North Central': 'W. N. Central',
}

cause_abbrev = {
    'equipment failure': 'Equipment Failure',
    'fuel supply emergency': 'Fuel Supply',
    'intentional attack': 'Intentional Attack',
    'islanding': 'Islanding',
    'public appeal': 'Public Appeal',
    'severe weather': 'Severe Weather',
    'system operability disruption': 'System Disruption',
}

pivot_table = pivot_table.rename(columns=region_abbrev, index=cause_abbrev)
pivot_table

CLIMATE.REGION,Central,E. N. Central,Northeast,Northwest,South,Southeast,Southwest,West,W. N. Central
CAUSE.CATEGORY,,,,,,,,,
Equipment Failure,322.0,26435.33,215.8,702.0,295.78,554.5,113.8,524.81,61.0
Fuel Supply,10035.25,33971.25,14629.57,1.0,16590.0,<NA>,76.0,6154.6,<NA>
Intentional Attack,346.06,2376.05,195.98,373.81,325.61,504.67,265.67,857.68,23.5
Islanding,125.33,1.0,881.0,73.33,493.5,<NA>,2.0,214.86,68.2
Public Appeal,1410.0,733.0,2655.0,898.0,1083.42,1895.67,2275.0,2028.11,439.5
Severe Weather,3273.49,4434.82,4429.9,4838.0,4391.35,2662.56,7378.78,2972.03,2442.5
System Disruption,2695.2,2610.0,773.5,141.0,866.07,169.31,329.22,363.67,<NA>


In [119]:
# Used LLM to fix table styling
styled_pivot_table = (
    pivot_table
    .style
    .background_gradient(cmap='Blues', axis=None)
    .format('{:,.0f}', na_rep='N/A')
    .set_properties(**{
        'text-align': 'center',
        'padding': '4px 8px',
        'font-size': '0.75rem',   # back to readable size, scale handles the rest
        'white-space': 'nowrap',
    })
    .set_table_styles([
        {'selector': 'th', 'props': [
            ('padding', '4px 8px'),
            ('font-size', '0.75rem'),
            ('white-space', 'nowrap'),
        ]},
        {'selector': 'table', 'props': [
            ('width', '100%'),
        ]},
    ])
)
# Export to HTML string
pivot_table_html = styled_pivot_table.to_html()

# Save to assets folder
with open('docs/assets/plots/pivot_table.html', 'w') as f:
    f.write(pivot_table_html)

## Step 3: Assessment of Missingness

In [121]:
df_cleaned['duration_missing'] = df_cleaned['OUTAGE.DURATION'].isna()
df_cleaned[df_cleaned['duration_missing']].to_csv('data/outages_with_missing_duration.csv', index=False)

In [ ]:
def calculate_tvd(df, col, target):
    pivoted = df.pivot_table(index=col, columns=target, aggfunc='size', fill_value=0, observed=True)
    dist = pivoted / pivoted.sum()
    return dist.diff(axis=1).iloc[:, -1].abs().sum() / 2

obs_tvd = calculate_tvd(df_cleaned, 'CLIMATE.REGION', 'duration_missing')

tvds = []
for _ in range(1000):
    shuffled = df_cleaned['CLIMATE.REGION'].sample(frac=1).reset_index(drop=True)
    temp_df = df_cleaned.assign(shuffled_region=shuffled)
    tvds.append(calculate_tvd(temp_df, 'shuffled_region', 'duration_missing'))

p_val_dep = np.mean(np.array(tvds) >= obs_tvd)
print(f"CLIMATE.REGION Observed TVD: {obs_tvd}, P-value: {p_val_dep}")

Observed TVD: 0.25449988607883345, P-value: 0.008


In [24]:
month_df = df_cleaned.dropna(subset=['MONTH']).copy()

obs_tvd_month = calculate_tvd(month_df, 'MONTH', 'duration_missing')

month_tvds = []
for _ in range(1000):
    shuffled_missing = month_df['duration_missing'].sample(frac=1).values

    temp_df = month_df.assign(shuffled_m = shuffled_missing)
    
    tvd = calculate_tvd(temp_df, 'MONTH', 'shuffled_m')
    month_tvds.append(tvd)

p_val_month = np.mean(np.array(month_tvds) >= obs_tvd_month)
print(f"MONTH Observed TVD: {obs_tvd_month:.4f}, P-value: {p_val_month:.4f}")

MONTH Observed TVD: 0.2219, P-value: 0.2470


In [122]:
clean_df = df_cleaned.dropna(subset=['ANOMALY.LEVEL']).copy()
m_obs = clean_df.loc[clean_df['duration_missing'] == True, 'ANOMALY.LEVEL'].mean()
nm_obs = clean_df.loc[clean_df['duration_missing'] == False, 'ANOMALY.LEVEL'].mean()
observed_diff = abs(m_obs - nm_obs)

diffs = []
for _ in range(1000):
    shuffled_labels = clean_df['duration_missing'].sample(frac=1, replace=False).values
    m = clean_df.loc[shuffled_labels, 'ANOMALY.LEVEL'].mean()
    nm = clean_df.loc[~shuffled_labels, 'ANOMALY.LEVEL'].mean()
    diffs.append(abs(m - nm))

p_val_anomaly = np.mean(np.array(diffs) >= observed_diff)
print(f"ANOMALY.LEVEL Observed Diff: {observed_diff:.4f}, P-value: {p_val_anomaly:.4f}")

fig_dist = ff.create_distplot(
    [diffs], 
    ['Shuffled Mean Differences'], 
    show_hist=True, 
    show_rug=False,
    bin_size=0.005
)

fig_dist.add_vline(
    x=observed_diff, 
    line_dash="dash", 
    line_color="red", 
    annotation_text=f"Observed Diff: {observed_diff:.4f}",
    annotation_position="top right"
)

fig_dist.update_layout(
    title="<b>Duration Missingness vs Anomaly Level</b><br>Empirical Distribution of the Test Statistic",
    xaxis_title="Absolute Difference in Mean Anomaly Level",
    yaxis_title="Density",
    template="plotly_white",
    bargap=0.02,
    legend=dict(
        orientation='h',      
        x=0.8,
        xanchor='center',
        y=1.05,            
        yanchor='bottom',
    ),
)

fig_dist.show()

# Export plot for website
fig_dist.write_html('docs/assets/plots/duration_missingness_vs_anomaly_level.html', include_plotlyjs='cdn')

ANOMALY.LEVEL Observed Diff: 0.7100, P-value: 0.0000


In [26]:
fig_box = px.box(
    df_cleaned, 
    x='duration_missing', 
    y='ANOMALY.LEVEL',
    color='duration_missing',
    notched=True,
    title='<b>Missingness Analysis: Anomaly Level Distribution</b><br>Duration Missing (True) vs. Not Missing (False)',
    labels={'duration_missing': 'Is Outage Duration Missing?', 'ANOMALY.LEVEL': 'Anomaly Level (ONI)'},
    template='plotly_white'
)

fig_box.show()

# Export plot for website
fig_box.write_html('docs/assets/plots/anomaly_level_by_duration_missingness.html', include_plotlyjs='cdn')

In [27]:
fig_month = px.box(
    df_cleaned, 
    x='duration_missing', 
    y='MONTH',
    color='duration_missing',
    notched=True,
    title='<b>Missingness Analysis: Month Distribution</b><br>Outage Duration Missing (True) vs. Not Missing (False)',
    labels={'duration_missing': 'Is Outage Duration Missing?', 'MONTH': 'Month (1-12)'},
    template='plotly_white',
    category_orders={'duration_missing': [False, True]}
)

# If the boxes are at the same level, it proves the missingness is MCAR with respect to time
fig_month.show()

# Export plot for website
fig_month.write_html('docs/assets/plots/month_by_duration_missingness.html', include_plotlyjs='cdn')

In [28]:
fig_dist_month = ff.create_distplot(
    [month_tvds], 
    ['Null Distribution (Shuffled Months)'], 
    show_hist=True, 
    show_rug=False,
    bin_size=0.005
)

fig_dist_month.add_vline(
    x=obs_tvd_month, 
    line_dash="dash", 
    line_color="red", 
    annotation_text=f"Observed TVD: {obs_tvd_month:.4f}"
)

fig_dist_month.update_layout(
    title="<b>Permutation Test Results: Month vs. Missingness</b>",
    xaxis_title="Total Variation Distance (TVD)",
    template="plotly_white", 
    bargap=0.02,
    legend=dict(
        orientation='h',      
        x=0.5,
        xanchor='center',
        y=1.05,            
        yanchor='bottom',
    ),
)

fig_dist_month.show()

# Export plot for website
fig_dist_month.write_html('docs/assets/plots/month_missingness_permutation_test.html', include_plotlyjs='cdn')


## Step 4: Hypothesis Testing

# Three options for hypothesis testing:

## Comparing Outage Durations by Cause (Permutation Test)

Our EDA shows that different causes lead to different outage times. We can test if this difference is statistically significant.

Null Hypothesis (H0): The distribution of outage durations is the same for outages caused by "severe weather" and "equipment failure." Any observed difference in sample means is due to random chance.

Alternative Hypothesis (H1): Outages caused by "severe weather" have a longer average duration than outages caused by "equipment failure."

Test Statistic: Difference in group means (Mean Duration of Severe Weather - Mean Duration of Equipment Failure).

## Climate Categories and Outage Causes (Chi-Square or TVD)

We have a CLIMATE.CATEGORY column (warm, cold, normal) and a CAUSE.CATEGORY column. We could see if the types of outages that occur depend on the climate conditions.

Null Hypothesis (H0): The distribution of outage causes is independent of the climate category (warm vs. cold).

Alternative Hypothesis (H1): The distribution of outage causes depends on the climate category (e.g., certain causes are more likely in cold climates vs. warm climates).

Test Statistic: Total Variation Distance (TVD) if you are comparing exactly two distributions, or the Chi-Square statistic if comparing across multiple categories.

## Customers Affected by Region (Permutation Test)

We can compare the impact of outages across different NERC or Climate regions.

Null Hypothesis (H0): Outages in the "Northeast" climate region affect the same number of customers, on average, as outages in the "South" climate region.

Alternative Hypothesis (H1): Outages in the "Northeast" climate region affect a different number of customers, on average, compared to the "South" climate region.

Test Statistic: Absolute difference in sample means ∣Mean Customers Northeast −Mean Customers South ∣. (Using the absolute difference makes this a two-sided test).

Test Statistic: Absolute difference in sample means ∣Mean Customers Northeast −Mean Customers South ∣. (Using the absolute difference makes this a two-sided test).

In [29]:
# Option 1: Severe Weather vs. Equipment Failure
mask = df_cleaned['CAUSE.CATEGORY'].isin(['severe weather', 'equipment failure'])
df_hyp = df_cleaned[mask].dropna(subset=['OUTAGE.DURATION']).copy()

means = df_hyp.groupby('CAUSE.CATEGORY', observed=True)['OUTAGE.DURATION'].mean()
obs_diff = means['severe weather'] - means['equipment failure']

diffs = []
for _ in range(1000):
    shuffled_causes = df_hyp['CAUSE.CATEGORY'].sample(frac=1, replace=False).values

    temp_df = df_hyp.assign(shuffled=shuffled_causes)
    shuffled_means = temp_df.groupby('shuffled', observed=True)['OUTAGE.DURATION'].mean()
    
    diffs.append(shuffled_means['severe weather'] - shuffled_means['equipment failure'])

p_val = np.mean(np.array(diffs) >= obs_diff)

print(f"Observed Difference: {obs_diff:.2f} minutes")
print(f"P-value: {p_val:.4f}")

Observed Difference: 2015.98 minutes
P-value: 0.0000


In [30]:
# Climate Category vs. Cause Category
climate_mask = df_cleaned['CLIMATE.CATEGORY'].isin(['warm', 'cold'])
df_hyp_b = df_cleaned[climate_mask].copy()

obs_tvd = calculate_tvd(df_hyp_b, 'CAUSE.CATEGORY', 'CLIMATE.CATEGORY')

tvd_diffs = []
for _ in range(1000):
    shuffled_climate = df_hyp_b['CLIMATE.CATEGORY'].sample(frac=1, replace=False).values

    temp_df = df_hyp_b.assign(shuffled_climate=shuffled_climate)

    tvd = calculate_tvd(temp_df, 'CAUSE.CATEGORY', 'shuffled_climate')
    tvd_diffs.append(tvd)

p_val_b = np.mean(np.array(tvd_diffs) >= obs_tvd)

print(f"Observed TVD: {obs_tvd:.4f}")
print(f"P-value: {p_val_b:.4f}")

Observed TVD: 0.0646
P-value: 0.3260


While our permutation test suggested that outage causes are independent of climate categories (p=0.3270), this may be due to the broad nature of the 'Climate Category' variable. Real-world events, like the Texas Power Crisis, suggest that localized extreme weather has a profound impact that might be masked when aggregating data at a national, multi-year level.

## Step 5: Framing a Prediction Problem

We would like to predict the duration of an outage given its early statistics when the incident occurs. As such, we will look to train a regression model and look at key statistics being RMSE, MAE, and R^2. 

There is also heavy skew in our target variable, with most outages lasting < 2 days. We will look to use the log distribution. 

In [31]:
# Outage Duration Distribution 
df_cleaned['OUTAGE.DURATION'].describe()

count      1468.0
mean      2582.58
std       5813.37
min           0.0
25%         100.0
50%         691.0
75%        2880.0
max      108653.0
Name: OUTAGE.DURATION, dtype: Float64

In [32]:
# Plot outage duration and log outage duration distributions side by side
fig = make_subplots(rows=1, cols=2)
fig.add_trace(
    go.Histogram(x=df_cleaned['OUTAGE.DURATION'], nbinsx=100, name='Outage Duration'),
    row=1, col=1
)
fig.add_trace(
    go.Histogram(x=np.log1p(df_cleaned['OUTAGE.DURATION']), nbinsx=30, name='Log Outage Duration'),
    row=1, col=2, 
)
fig.update_layout(
    title='<b>Outage Duration Distribution</b><br>Original vs. Log-Transformed',
    xaxis_title='Outage Duration (Minutes)',
    xaxis2_title='Log Outage Duration',
    yaxis_title='Count',
    template='plotly_white',
    showlegend=False, 
    bargap=0.02
)
fig.show()

# Export plot for website
fig.write_html('docs/assets/plots/outage_duration_distribution.html', include_plotlyjs='cdn')

### Further investigation on missing rows
Previous dataframe was cleaned for preliminary analysis and EDA, we will look to further investigate missing data and perform imputation accordingly. End result will be our engineered dataframe with key features for model training and analysis. 

In [33]:
# All missing rows in "cleaned" dataset: 
missing_count = df_cleaned.isna().sum()[df_cleaned.isna().sum() > 0].sort_values(ascending=False)
missing_count

HURRICANE.NAMES            1454
DEMAND.LOSS.MW              700
CAUSE.CATEGORY.DETAIL       467
CUSTOMERS.AFFECTED          437
OUTAGE.RESTORATION           58
OUTAGE.RESTORATION.DATE      58
OUTAGE.RESTORATION.TIME      58
OUTAGE.DURATION              58
IND.PRICE                    22
TOTAL.SALES                  22
IND.SALES                    22
COM.SALES                    22
RES.SALES                    22
TOTAL.PRICE                  22
COM.PRICE                    22
IND.PERCEN                   22
RES.PRICE                    22
COM.PERCEN                   22
RES.PERCEN                   22
POPDEN_UC                    10
POPDEN_RURAL                 10
OUTAGE.START                  9
MONTH                         9
OUTAGE.START.TIME             9
OUTAGE.START.DATE             9
CLIMATE.CATEGORY              9
ANOMALY.LEVEL                 9
MONTH.NAME                    9
CLIMATE.REGION                6
dtype: int64

In [34]:
# CAUSE.CATEGORY.DETAIL has 467 missing values, can investigate CAUSE.CATEGORY to impute missing values
print(df_cleaned['CAUSE.CATEGORY'].value_counts())

# CAUSE.CATEGORY of missing CAUSE.CATEGORY.DETAIL values
print(f"\n---> Filtered by missing CAUSE.CATEGORY.DETAIL\n{df_cleaned[df_cleaned['CAUSE.CATEGORY.DETAIL'].isna()]['CAUSE.CATEGORY'].value_counts()}")

CAUSE.CATEGORY
severe weather                   760
intentional attack               418
system operability disruption    127
public appeal                     65
equipment failure                 60
fuel supply emergency             50
islanding                         46
Name: count, dtype: int64

---> Filtered by missing CAUSE.CATEGORY.DETAIL
CAUSE.CATEGORY
severe weather                   187
system operability disruption     90
public appeal                     65
intentional attack                48
islanding                         46
fuel supply emergency             19
equipment failure                 12
Name: count, dtype: int64


In [35]:
# Investigate missing clinmate regions --> Hawaii and Alaska only
df_cleaned[df_cleaned['CLIMATE.REGION'].isna()]

,YEAR,MONTH,U.S._STATE,POSTAL.CODE,NERC.REGION,CLIMATE.REGION,ANOMALY.LEVEL,CLIMATE.CATEGORY,OUTAGE.START.DATE,OUTAGE.START.TIME,OUTAGE.RESTORATION.DATE,OUTAGE.RESTORATION.TIME,CAUSE.CATEGORY,CAUSE.CATEGORY.DETAIL,HURRICANE.NAMES,OUTAGE.DURATION,DEMAND.LOSS.MW,CUSTOMERS.AFFECTED,RES.PRICE,COM.PRICE,IND.PRICE,TOTAL.PRICE,RES.SALES,COM.SALES,IND.SALES,TOTAL.SALES,RES.PERCEN,COM.PERCEN,IND.PERCEN,RES.CUSTOMERS,COM.CUSTOMERS,IND.CUSTOMERS,TOTAL.CUSTOMERS,RES.CUST.PCT,COM.CUST.PCT,IND.CUST.PCT,PC.REALGSP.STATE,PC.REALGSP.USA,PC.REALGSP.REL,PC.REALGSP.CHANGE,UTIL.REALGSP,TOTAL.REALGSP,UTIL.CONTRI,PI.UTIL.OFUSA,POPULATION,POPPCT_URBAN,POPPCT_UC,POPDEN_URBAN,POPDEN_UC,POPDEN_RURAL,AREAPCT_URBAN,AREAPCT_UC,PCT_LAND,PCT_WATER_TOT,PCT_WATER_INLAND,OUTAGE.START,OUTAGE.RESTORATION,MONTH.NAME,duration_missing
1516,2008,12,Hawaii,HI,HI,NaN,-0.7,cold,2008-12-26,18:13:00,2008-12-27,17:00:00,severe weather,thunderstorm,NaN,1367,1060,294000,29.28,26.28,22.43,25.78,253625.0,275383.0,306397.0,835405.0,30.36,32.96,36.68,409668.0,61684.0,673.0,472025.0,86.79,13.07,0.14,50565.0,48401.0,1.04,-0.7,1646.0,67364.0,2.44,0.5,1332213,91.93,20.47,3180.8,1664.7,18.2,6.12,2.60,58.75,41.25,0.38,2008-12-26 18:13:00,2008-12-27 17:00:00,Dec,False
1517,2011,5,Hawaii,HI,PR,NaN,-0.4,normal,2011-05-02,17:06:00,2011-05-02,20:00:00,severe weather,NaN,NaN,174,220,62000,34.58,32.14,27.85,31.29,249369.0,292304.0,310682.0,852355.0,29.26,34.29,36.45,417531.0,60043.0,698.0,478272.0,87.30,12.55,0.15,49110.0,47586.0,1.03,0.4,1935.0,67686.0,2.86,0.6,1378227,91.93,20.47,3180.8,1664.7,18.2,6.12,2.60,58.75,41.25,0.38,2011-05-02 17:06:00,2011-05-02 20:00:00,May,False
1518,2006,10,Hawaii,HI,HECO,NaN,0.7,warm,2006-10-15,07:09:00,2006-10-15,16:12:00,severe weather,earthquake,NaN,543,110,59886,23.25,21.26,17.71,20.54,274609.0,307570.0,340735.0,922915.0,29.75,33.33,36.92,401592.0,61334.0,689.0,463615.0,86.62,13.23,0.15,50358.0,48909.0,1.03,1.1,1436.0,65956.0,2.18,0.5,1309731,91.93,20.47,3180.8,1664.7,18.2,6.12,2.60,58.75,41.25,0.38,2006-10-15 07:09:00,2006-10-15 16:12:00,Oct,False
1519,2006,6,Hawaii,HI,HECO,NaN,0.0,normal,2006-06-01,14:12:00,2006-06-01,18:09:00,system operability disruption,NaN,NaN,237,120,29300,24.09,22.06,18.74,21.45,265474.0,298487.0,327618.0,891580.0,29.78,33.48,36.75,401592.0,61334.0,689.0,463615.0,86.62,13.23,0.15,50358.0,48909.0,1.03,1.1,1436.0,65956.0,2.18,0.5,1309731,91.93,20.47,3180.8,1664.7,18.2,6.12,2.60,58.75,41.25,0.38,2006-06-01 14:12:00,2006-06-01 18:09:00,Jun,False
1520,2006,10,Hawaii,HI,HECO,NaN,0.7,warm,2006-10-15,07:09:00,2006-10-16,14:55:00,severe weather,earthquake,NaN,1906,1170,291000,23.25,21.26,17.71,20.54,274609.0,307570.0,340735.0,922915.0,29.75,33.33,36.92,401592.0,61334.0,689.0,463615.0,86.62,13.23,0.15,50358.0,48909.0,1.03,1.1,1436.0,65956.0,2.18,0.5,1309731,91.93,20.47,3180.8,1664.7,18.2,6.12,2.60,58.75,41.25,0.38,2006-10-15 07:09:00,2006-10-16 14:55:00,Oct,False
1534,2000,<NA>,Alaska,AK,ASCC,NaN,NaN,NaN,NaT,NaN,NaT,NaN,equipment failure,failure,NaN,<NA>,35,14273,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,230534.0,38074.0,854.0,273530.0,84.28,13.92,0.31,57401.0,44745.0,1.28,-2.2,724.0,36046.0,2.01,0.2,627963,66.02,21.56,1802.6,1276.0,0.4,0.05,0.02,85.76,14.24,2.90,NaT,NaT,NaN,True


In [36]:
# Can just use CAUSE.CATEGORY.DETAIL since it includes hurricane information
hurricanes_detail = (df_cleaned['CAUSE.CATEGORY.DETAIL'] == 'hurricanes')
no_names = (df_cleaned['HURRICANE.NAMES'].isna())
len(df_cleaned[~no_names]), len(df_cleaned[hurricanes_detail]), len(df_cleaned[hurricanes_detail & no_names])

(72, 74, 2)

In [37]:
# Imputations and Feature engineering functions

def impute_climate_region(df):
    # Hawaii and Alaska don't have climate regions in dataset. 
    df.loc[df['U.S._STATE'] == 'Hawaii', 'CLIMATE.REGION'] = 'Hawaii'
    df.loc[df['U.S._STATE'] == 'Alaska', 'CLIMATE.REGION'] = 'Alaska'
    return df

def impute_cause_category_detail(df):
    # Impute based on CAUSE.CATEGORY
    fallback = {
        'severe weather':                  'Other Weather',
        'intentional attack':              'Other Intentional Attack',
        'system operability disruption':   'Other Equipment Failure',
        'equipment failure':               'Other Equipment Failure',
        'fuel supply emergency':           'Other Fuel Supply Emergency',
        'public appeal':                   'Other Public Appeal',
        'islanding':                       'Other Islanding',
    }

    df['CAUSE.CATEGORY.DETAIL'] = df['CAUSE.CATEGORY.DETAIL'].fillna(
        df['CAUSE.CATEGORY'].map(fallback)
    )
    return df

def drop_missing_months(df):
    df = df.dropna(subset=['OUTAGE.START'])
    return df

def drop_missing_price(df):
    df = df.dropna(subset=['RES.PRICE'])
    return df

def median_impute_customers_affected(df): 
    # Median imputation for CUSTOMERS.AFFECTED, grouped by CAUSE.CATEGORY
    df['CUSTOMERS.AFFECTED'] = df.groupby('CAUSE.CATEGORY')['CUSTOMERS.AFFECTED'].transform(
        lambda x: x.fillna(x.median().round())
    )
    return df

def drop_missing_durations(df):
    df = df.dropna(subset=['OUTAGE.DURATION'])
    return df

def identify_weekend(df): 
    df['OUTAGE.START'] = pd.to_datetime(df['OUTAGE.START'])
    df['START_ON_WEEKEND'] = df['OUTAGE.START'].dt.weekday.apply(lambda x: 1 if x >= 5 else 0) # 1 for Weekends, 0 for weekdays
    return df

def identify_season(df): 
    outage_month = pd.to_datetime(df['OUTAGE.START']).dt.month
    df['SEASON'] = outage_month.map({
            12: 'Winter', 1: 'Winter',  2: 'Winter',
            3:  'Spring', 4: 'Spring',  5: 'Spring',
            6:  'Summer', 7: 'Summer',  8: 'Summer',
            9:  'Fall',   10: 'Fall',   11: 'Fall',
        })
    return df

def cyclical_encode_month(df):
    # Cyclical encoding for MONTH
    df['MONTH_SIN'] = np.sin(2 * np.pi * df['MONTH'] / 12)
    df['MONTH_COS'] = np.cos(2 * np.pi * df['MONTH'] / 12)
    return df

def remove_outage_outliers(df):
    # Remove extreme outliers in OUTAGE.DURATION (e.g., above 30,000 minutes or ~20 days)
    df = df[df['OUTAGE.DURATION'] <= 30000]
    return df

In [38]:
df_engineered = (
    df_cleaned.pipe(impute_climate_region)
      .pipe(impute_cause_category_detail)
      .pipe(drop_missing_months)
      .pipe(drop_missing_price)
      .pipe(drop_missing_durations)
      .pipe(median_impute_customers_affected)
      .pipe(identify_weekend)
      .pipe(identify_season)
      .pipe(cyclical_encode_month)
      .pipe(remove_outage_outliers)
)

print(f"Rows dropped: {len(df_cleaned) - len(df_engineered)} ({(len(df_cleaned) - len(df_engineered)) / len(df_cleaned) * 100:.1f}% of original)")

Rows dropped: 77 (5.0% of original)


In [39]:
df_engineered.head()

,YEAR,MONTH,U.S._STATE,POSTAL.CODE,NERC.REGION,CLIMATE.REGION,ANOMALY.LEVEL,CLIMATE.CATEGORY,OUTAGE.START.DATE,OUTAGE.START.TIME,OUTAGE.RESTORATION.DATE,OUTAGE.RESTORATION.TIME,CAUSE.CATEGORY,CAUSE.CATEGORY.DETAIL,HURRICANE.NAMES,OUTAGE.DURATION,DEMAND.LOSS.MW,CUSTOMERS.AFFECTED,RES.PRICE,COM.PRICE,IND.PRICE,TOTAL.PRICE,RES.SALES,COM.SALES,IND.SALES,TOTAL.SALES,RES.PERCEN,COM.PERCEN,IND.PERCEN,RES.CUSTOMERS,COM.CUSTOMERS,IND.CUSTOMERS,TOTAL.CUSTOMERS,RES.CUST.PCT,COM.CUST.PCT,IND.CUST.PCT,PC.REALGSP.STATE,PC.REALGSP.USA,PC.REALGSP.REL,PC.REALGSP.CHANGE,UTIL.REALGSP,TOTAL.REALGSP,UTIL.CONTRI,PI.UTIL.OFUSA,POPULATION,POPPCT_URBAN,POPPCT_UC,POPDEN_URBAN,POPDEN_UC,POPDEN_RURAL,AREAPCT_URBAN,AREAPCT_UC,PCT_LAND,PCT_WATER_TOT,PCT_WATER_INLAND,OUTAGE.START,OUTAGE.RESTORATION,MONTH.NAME,duration_missing,START_ON_WEEKEND,SEASON,MONTH_SIN,MONTH_COS
1,2011,7,Minnesota,MN,MRO,East North Central,-0.3,normal,2011-07-01,17:00:00,2011-07-03,20:00:00,severe weather,Other Weather,NaN,3060,<NA>,70000,11.60,9.18,6.81,9.28,2.33e+06,2.11e+06,2.11e+06,6.56e+06,35.55,32.23,32.20,2.31e+06,276286.0,10673.0,2.60e+06,88.94,10.64,0.41,51268.0,47586.0,1.08,1.6,4802.0,274182.0,1.75,2.2,5348119,73.27,15.28,2279.0,1700.5,18.2,2.14,0.6,91.59,8.41,5.48,2011-07-01 17:00:00,2011-07-03 20:00:00,Jul,False,0,Summer,-0.5,-0.87
2,2014,5,Minnesota,MN,MRO,East North Central,-0.1,normal,2014-05-11,18:38:00,2014-05-11,18:39:00,intentional attack,vandalism,NaN,1,<NA>,0,12.12,9.71,6.49,9.28,1.59e+06,1.81e+06,1.89e+06,5.28e+06,30.03,34.21,35.73,2.35e+06,284978.0,9898.0,2.64e+06,88.83,10.79,0.37,53499.0,49091.0,1.09,1.9,5226.0,291955.0,1.79,2.2,5457125,73.27,15.28,2279.0,1700.5,18.2,2.14,0.6,91.59,8.41,5.48,2014-05-11 18:38:00,2014-05-11 18:39:00,May,False,1,Spring,0.5,-0.87
3,2010,10,Minnesota,MN,MRO,East North Central,-1.5,cold,2010-10-26,20:00:00,2010-10-28,22:00:00,severe weather,heavy wind,NaN,3000,<NA>,70000,10.87,8.19,6.07,8.15,1.47e+06,1.80e+06,1.95e+06,5.22e+06,28.10,34.50,37.37,2.30e+06,276463.0,10150.0,2.59e+06,88.92,10.69,0.39,50447.0,47287.0,1.07,2.7,4571.0,267895.0,1.71,2.1,5310903,73.27,15.28,2279.0,1700.5,18.2,2.14,0.6,91.59,8.41,5.48,2010-10-26 20:00:00,2010-10-28 22:00:00,Oct,False,0,Fall,-0.87,0.5
4,2012,6,Minnesota,MN,MRO,East North Central,-0.1,normal,2012-06-19,04:30:00,2012-06-20,23:00:00,severe weather,thunderstorm,NaN,2550,<NA>,68200,11.79,9.25,6.71,9.19,1.85e+06,1.94e+06,1.99e+06,5.79e+06,31.99,33.54,34.44,2.32e+06,278466.0,11010.0,2.61e+06,88.90,10.68,0.42,51598.0,48156.0,1.07,0.6,5364.0,277627.0,1.93,2.2,5380443,73.27,15.28,2279.0,1700.5,18.2,2.14,0.6,91.59,8.41,5.48,2012-06-19 04:30:00,2012-06-20 23:00:00,Jun,False,0,Summer,0.0,-1.0
5,2015,7,Minnesota,MN,MRO,East North Central,1.2,warm,2015-07-18,02:00:00,2015-07-19,07:00:00,severe weather,Other Weather,NaN,1740,250,250000,13.07,10.16,7.74,10.43,2.03e+06,2.16e+06,1.78e+06,5.97e+06,33.98,36.21,29.78,2.37e+06,289044.0,9812.0,2.67e+06,88.82,10.81,0.37,54431.0,49844.0,1.09,1.7,4873.0,292023.0,1.67,2.2,5489594,73.27,15.28,2279.0,1700.5,18.2,2.14,0.6,91.59,8.41,5.48,2015-07-18 02:00:00,2015-07-19 07:00:00,Jul,False,1,Summer,-0.5,-0.87


In [40]:
# Some columns will still have missing values, but our selected features will only be using a subset
df_engineered.isna().sum()[df_engineered.isna().sum() > 0] 

HURRICANE.NAMES    1378
DEMAND.LOSS.MW      660
POPDEN_UC            10
POPDEN_RURAL         10
dtype: int64

## Feature setup and Training functions

In [41]:
# Feature selection

cat_features = [
    'SEASON', 
    'U.S._STATE', 
    'NERC.REGION', 
    'CLIMATE.REGION', 
    'CLIMATE.CATEGORY',
    'CAUSE.CATEGORY',
    'CAUSE.CATEGORY.DETAIL',
]
num_features = [
    'MONTH_SIN', 
    'MONTH_COS',
    'START_ON_WEEKEND',
    'ANOMALY.LEVEL', 
    'CUSTOMERS.AFFECTED',
    'RES.PRICE',
    'COM.PRICE',
    'IND.PRICE',
    'UTIL.CONTRI',
    'PI.UTIL.OFUSA',
]

# Preprocessing Pipeline
preprocessor = ColumnTransformer(
    transformers=[
        ('num', StandardScaler(), num_features),
        ('cat', OneHotEncoder(handle_unknown='ignore'), cat_features)
    ]
)

preprocessor_dense = ColumnTransformer(
    transformers=[
        ('num', StandardScaler(), num_features),
        ('cat', OneHotEncoder(handle_unknown='ignore', sparse_output=False), cat_features)
    ]
)

features = cat_features + num_features
target = 'OUTAGE.DURATION'

In [42]:
print(f"Initial cleaned dataset shape:  {df.shape}")
print(f"Engineered dataset shape:       {df_engineered.shape}")

X = df_engineered[features]
y = np.log1p(df_engineered[target]) # Log transform target to reduce skew
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

print(f"Training set shape:             {X_train.shape}, {y_train.shape}")
print(f"Test set shape:                 {X_test.shape}, {y_test.shape}")

Initial cleaned dataset shape:  (1534, 55)
Engineered dataset shape:       (1449, 63)
Training set shape:             (1159, 17), (1159,)
Test set shape:                 (290, 17), (290,)


In [43]:
# General Setup for testing different models

results_log = [] # For final dataframe creation

def plot_log_and_reg_residuals(residual_df, model_name):
    fig = make_subplots(rows=1, cols=2, subplot_titles=[
        'Residuals vs Predicted (minutes)',
        'Residuals vs Predicted (log scale)',
    ])
    fig.add_trace(
        go.Scatter(
            x=residual_df['Predicted Duration (min)'],
            y=residual_df['Residuals (min)'],
            mode='markers', opacity=0.5, marker=dict(size=4),
            name='minutes',
        ),
        row=1, col=1,
    )
    fig.add_trace(
        go.Scatter(
            x=residual_df['Predicted Log Duration'],
            y=residual_df['Residuals (log)'],
            mode='markers', opacity=0.5, marker=dict(size=4),
            name='log scale',
        ),
        row=1, col=2,
    )


    # Zero lines for reference
    fig.add_hline(y=0, line_dash='dash', line_color='black', row=1, col=1)
    fig.add_hline(y=0, line_dash='dash', line_color='black', row=1, col=2)

    fig.update_layout(
        title=f'<b>Residuals - {model_name}</b>',
        title_pad=dict(t=20),
        margin=dict(t=80),
        autosize=True,
        height=450, # seems best for visibility
        showlegend=False,
        template='plotly_white'
    )
    fig.update_xaxes(title_text='Predicted (log min)', row=1, col=2)
    fig.update_xaxes(title_text='Predicted (min)', row=1, col=1)
    
    # Set y-axis limits to be symmetric around zero and slightly larger than max absolute residual 
    abs_max_min = residual_df['Residuals (min)'].abs().max() * 1.1
    abs_max_log = residual_df['Residuals (log)'].abs().max() * 1.1

    fig.update_yaxes(title_text='Residuals', range=[-abs_max_min, abs_max_min], row=1, col=1)
    fig.update_yaxes(title_text='Residuals', range=[-abs_max_log, abs_max_log], row=1, col=2)    
    fig.show()
    return fig

def train_and_evaluate_model(model, X_train, y_train, X_test, y_test, label=None):
    start_time = time.time()
    model.fit(X_train, y_train)
    y_pred = model.predict(X_test)

    y_pred_minutes = np.expm1(y_pred) # Convert back to minutes for residuals plot
    y_test_minutes = np.expm1(y_test)

    metrics = {
        'Model': label or model.__class__.__name__,
        'MAE (log)': mean_absolute_error(y_test, y_pred),
        'RMSE (log)': root_mean_squared_error(y_test, y_pred),
        'MAE (minutes)': mean_absolute_error(y_test_minutes, y_pred_minutes),
        'RMSE (minutes)': root_mean_squared_error(y_test_minutes, y_pred_minutes),
        'R^2': r2_score(y_test, y_pred),           
    }

    print(f"Model:                   {metrics['Model']}")
    print(f"MAE (log):               {metrics['MAE (log)']:.4f}")
    print(f"RMSE (log):              {metrics['RMSE (log)']:.4f}")
    print(f"MAE (minutes):           {metrics['MAE (minutes)']:.2f} minutes")
    print(f"RMSE (minutes):          {metrics['RMSE (minutes)']:.2f} minutes")
    print(f"R^2:                     {metrics['R^2']:.4f}")
    print(f"Training Time:           {time.time() - start_time:.2f} seconds")
    # Plot residuals
    residual_df = pd.DataFrame({
        'Predicted Log Duration': y_pred,
        'Residuals (log)': y_test - y_pred,
        'Predicted Duration (min)': y_pred_minutes,
        'Residuals (min)': y_test_minutes - y_pred_minutes
    })
    fig = plot_log_and_reg_residuals(residual_df, metrics['Model'])

    results_log.append(metrics)
    return model, y_pred, fig

def train_tune_evaluate(model, param_distributions, X_train, y_train, X_test, y_test, n_iter=50, label=None):
    random_search = RandomizedSearchCV(
        estimator=model,
        param_distributions=param_distributions,
        n_iter=n_iter,
        scoring='neg_mean_absolute_error',
        cv=3,
        verbose=1,
        random_state=42,
        n_jobs=-1
    )
    print(f"Starting hyperparameter tuning for {label or model.__class__.__name__}...")
    start_time = time.time()
    random_search.fit(X_train, y_train)
    print(f"Completed in {time.time() - start_time:.2f} seconds")

    print(f"Best Hyperparameters: {random_search.best_params_}")
    best_model = random_search.best_estimator_

    tuned_label = f"{label or model.__class__.__name__} (tuned)"
    return train_and_evaluate_model(best_model, X_train, y_train, X_test, y_test, label=tuned_label)

## Step 6: Baseline Model

Use naive linear regression, without any tuning

In [44]:
# Linear Regression - Baseline
linear_pipeline = Pipeline(steps=[
    ('preprocessor', preprocessor),
    ('regressor', LinearRegression())
])

linear_reg, linear_pred, linear_fig = train_and_evaluate_model(linear_pipeline, X_train, y_train, X_test, y_test, label="Linear Regression (Baseline)")

# Export plot for website
linear_fig.write_html('docs/assets/plots/linear_regression_residuals.html', include_plotlyjs='cdn')

Model:                   Linear Regression (Baseline)
MAE (log):               1.4645
RMSE (log):              1.9254
MAE (minutes):           1812.06 minutes
RMSE (minutes):          4378.00 minutes
R^2:                     0.5185
Training Time:           0.02 seconds


## Step 7: Final Model

In [45]:
# Ridge Regression - Hyperparameter Tuning
ridge_pipeline = Pipeline(steps=[
    ('preprocessor', preprocessor),
    ('regressor', Ridge())
])
ridge_param_dist = {
    'regressor__alpha': np.logspace(-3, 3, 50)
}
ridge_reg, ridge_pred, ridge_fig = train_tune_evaluate(ridge_pipeline, ridge_param_dist, X_train, y_train, X_test, y_test, n_iter=1000,label="Ridge Regression")

# Export plot for website
ridge_fig.write_html('docs/assets/plots/ridge_regression_residuals.html', include_plotlyjs='cdn')

Starting hyperparameter tuning for Ridge Regression...
Fitting 3 folds for each of 50 candidates, totalling 150 fits


c:\ProgramData\anaconda3\Lib\site-packages\sklearn\model_selection\_search.py:320: UserWarning:

The total space of parameters 50 is smaller than n_iter=1000. Running 50 iterations. For exhaustive searches, use GridSearchCV.



Completed in 2.39 seconds
Best Hyperparameters: {'regressor__alpha': 8.286427728546842}
Model:                   Ridge Regression (tuned)
MAE (log):               1.4789
RMSE (log):              1.9142
MAE (minutes):           1802.69 minutes
RMSE (minutes):          3925.18 minutes
R^2:                     0.5240
Training Time:           0.01 seconds


In [46]:
# Random Forest Regressor
rf_pipeline = Pipeline(steps=[
    ('preprocessor', preprocessor_dense),
    ('regressor', RandomForestRegressor(random_state=42))
])
rf_param_dist = {
    'regressor__n_estimators': randint(100, 1000),
    'regressor__max_depth': [None, 1, 2, 3, 4, 5, 10, 20, 30, 40, 50],
    'regressor__min_samples_split': randint(2, 30),
    'regressor__min_samples_leaf': randint(1, 20),
    'regressor__max_features': ['sqrt', 'log2', 0.3, 0.5, 0.7, 1.0],
    'regressor__bootstrap': [True, False],
    'regressor__oob_score': [True, False],
    'regressor__warm_start': [True, False],
    'regressor__ccp_alpha': uniform(0.0, 0.01),
    'regressor__max_samples': [0.6, 0.7, 0.8, 0.9, None],
}
rf_reg, rf_pred, rf_fig = train_tune_evaluate(rf_pipeline, rf_param_dist, X_train, y_train, X_test, y_test, n_iter=500, label="Random Forest Regressor")

# Export plot for website
rf_fig.write_html('docs/assets/plots/random_forest_residuals.html', include_plotlyjs='cdn')

Starting hyperparameter tuning for Random Forest Regressor...
Fitting 3 folds for each of 500 candidates, totalling 1500 fits


c:\ProgramData\anaconda3\Lib\site-packages\sklearn\model_selection\_validation.py:540: FitFailedWarning:


711 fits failed out of a total of 1500.
The score on these train-test partitions for these parameters will be set to nan.
If these failures are not expected, you can try to debug them by setting error_score='raise'.

Below are more details about the failures:
--------------------------------------------------------------------------------
624 fits failed with the following error:
Traceback (most recent call last):
  File "c:\ProgramData\anaconda3\Lib\site-packages\sklearn\model_selection\_validation.py", line 888, in _fit_and_score
    estimator.fit(X_train, y_train, **fit_params)
  File "c:\ProgramData\anaconda3\Lib\site-packages\sklearn\base.py", line 1473, in wrapper
    return fit_method(estimator, *args, **kwargs)
           ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "c:\ProgramData\anaconda3\Lib\site-packages\sklearn\pipeline.py", line 473, in fit
    self._final_estimato

Completed in 52.61 seconds
Best Hyperparameters: {'regressor__bootstrap': False, 'regressor__ccp_alpha': 0.005464303160778473, 'regressor__max_depth': None, 'regressor__max_features': 0.3, 'regressor__max_samples': None, 'regressor__min_samples_leaf': 1, 'regressor__min_samples_split': 6, 'regressor__n_estimators': 382, 'regressor__oob_score': False, 'regressor__warm_start': False}
Model:                   Random Forest Regressor (tuned)
MAE (log):               1.3368
RMSE (log):              1.8286
MAE (minutes):           1739.02 minutes
RMSE (minutes):          3719.70 minutes
R^2:                     0.5657
Training Time:           1.32 seconds


In [47]:
# XGBoost Regressor
xgb_pipeline = Pipeline(steps=[
    ('preprocessor', preprocessor_dense),
    ('regressor', XGBRegressor(random_state=42, objective='reg:squarederror'))
])

xgb_param_dist = {
    'regressor__n_estimators': randint(100, 1000),
    'regressor__learning_rate': uniform(0.01, 0.2),
    'regressor__max_depth': randint(3, 15),
    'regressor__subsample': uniform(0.6, 0.4),
    'regressor__colsample_bytree': uniform(0.4, 0.6),
    'regressor__min_child_weight': randint(1, 10),
    'regressor__gamma': uniform(0, 0.5),
    'regressor__reg_alpha': uniform(0, 1),
    'regressor__reg_lambda': uniform(0.5, 2),
}

xgb_reg, xgb_pred, xgb_fig = train_tune_evaluate(xgb_pipeline, xgb_param_dist, X_train, y_train, X_test, y_test, n_iter=1000, label="XGBoost Regressor")

# Export plot for website
xgb_fig.write_html('docs/assets/plots/xgboost_residuals.html', include_plotlyjs='cdn')

Starting hyperparameter tuning for XGBoost Regressor...
Fitting 3 folds for each of 1000 candidates, totalling 3000 fits
Completed in 61.82 seconds
Best Hyperparameters: {'regressor__colsample_bytree': 0.44895923200290305, 'regressor__gamma': 0.129295795567392, 'regressor__learning_rate': 0.015570448164214882, 'regressor__max_depth': 14, 'regressor__min_child_weight': 2, 'regressor__n_estimators': 420, 'regressor__reg_alpha': 0.6728471334885447, 'regressor__reg_lambda': 1.9485090549982664, 'regressor__subsample': 0.7970396020887256}
Model:                   XGBoost Regressor (tuned)
MAE (log):               1.3745
RMSE (log):              1.8994
MAE (minutes):           1755.67 minutes
RMSE (minutes):          3729.29 minutes
R^2:                     0.5314
Training Time:           0.43 seconds


In [48]:
# HistGradientBoostingRegressor, typically better for larger datasets, but can still be competitive here 
hgb_pipeline = Pipeline(steps=[
    ('preprocessor', preprocessor_dense),
    ('regressor', HistGradientBoostingRegressor(random_state=42))
])

hgb_param_dist = {
    'regressor__learning_rate':    uniform(0.01, 0.2),
    'regressor__max_iter':         randint(100, 500),
    'regressor__max_depth':        randint(3, 10),
    'regressor__min_samples_leaf': randint(10, 50),
    'regressor__l2_regularization': uniform(0, 1),
    'regressor__max_leaf_nodes':   randint(20, 100),
}

hgb_reg, hgb_pred, hgb_fig = train_tune_evaluate(hgb_pipeline, hgb_param_dist, X_train, y_train, X_test, y_test, n_iter=1000, label="HistGradientBoostingRegressor")

# Export plot for website
hgb_fig.write_html('docs/assets/plots/hist_gradient_boosting_residuals.html', include_plotlyjs='cdn')


Starting hyperparameter tuning for HistGradientBoostingRegressor...
Fitting 3 folds for each of 1000 candidates, totalling 3000 fits
Completed in 86.51 seconds
Best Hyperparameters: {'regressor__l2_regularization': 0.4395116886329101, 'regressor__learning_rate': 0.031132936852348782, 'regressor__max_depth': 8, 'regressor__max_iter': 259, 'regressor__max_leaf_nodes': 62, 'regressor__min_samples_leaf': 11}
Model:                   HistGradientBoostingRegressor (tuned)
MAE (log):               1.3421
RMSE (log):              1.8624
MAE (minutes):           1691.86 minutes
RMSE (minutes):          3557.68 minutes
R^2:                     0.5495
Training Time:           0.49 seconds


In [53]:
# Additional investment into XGBoost with Optuna

from optuna.samplers import TPESampler
from optuna.pruners import MedianPruner

def objective(trial):
    param = {
        'n_estimators': trial.suggest_int('n_estimators', 100, 1000),
        'learning_rate': trial.suggest_float('learning_rate', 0.01, 0.3, log=True),
        'max_depth': trial.suggest_int('max_depth', 2, 12),
        'subsample': trial.suggest_float('subsample', 0.5, 1.0),
        'colsample_bytree': trial.suggest_float('colsample_bytree', 0.4, 1.0),
        'min_child_weight': trial.suggest_int('min_child_weight', 1, 10),
        'gamma': trial.suggest_float('gamma', 1e-8, 1.0, log=True),
        'reg_alpha': trial.suggest_float('reg_alpha', 1e-8, 1.0, log=True),
        'reg_lambda': trial.suggest_float('reg_lambda', 1e-8, 1.0, log=True),
        'tree_method': 'hist', 
    }

    pipeline = Pipeline(steps=[
        ('preprocessor', preprocessor_dense),
        ('regressor', XGBRegressor(
            random_state=42,
            objective='reg:squarederror',
            **param
        ))
    ])

    scores = cross_val_score(
        pipeline, X_train, y_train,
        cv=3,
        scoring='neg_mean_absolute_error',
        n_jobs=-1  
    )
    return scores.mean()

# Run the Optuna study
print("Starting Optuna study for XGBoost")
start_time = time.time()

study = optuna.create_study(
    direction='maximize',
    sampler=TPESampler(seed=42, n_startup_trials=50),
    pruner=MedianPruner(n_startup_trials=10, n_warmup_steps=1),
)
study.optimize(objective, n_trials=1000)

print(f"Best Optuna parameters: {study.best_params}")

# Train the final model with best params
best_xgb_pipeline = Pipeline(steps=[
    ('preprocessor', preprocessor_dense),
    ('regressor', XGBRegressor(
        random_state=42,
        objective='reg:squarederror',
        **study.best_params
    ))
])

print("\n\nTuning completed in {:.2f} seconds".format(time.time() - start_time))
optuna_xgb, optuna_xgb_pred, optuna_xgb_fig = train_and_evaluate_model(best_xgb_pipeline, X_train, y_train, X_test, y_test, label="XGBoost (Optuna tuned)")

# Export plot for website
optuna_xgb_fig.write_html('docs/assets/plots/xgboost_optuna_residuals.html', include_plotlyjs='cdn')

[I 2026-03-13 19:09:24,577] A new study created in memory with name: no-name-b877beff-c977-45f5-b878-b78415417996


Starting Optuna study for XGBoost


[I 2026-03-13 19:09:26,369] Trial 0 finished with value: -1.45118023753422 and parameters: {'n_estimators': 437, 'learning_rate': 0.2536999076681772, 'max_depth': 10, 'subsample': 0.7993292420985183, 'colsample_bytree': 0.4936111842654619, 'min_child_weight': 2, 'gamma': 2.9152036385288193e-08, 'reg_alpha': 0.08499808989182997, 'reg_lambda': 0.0006440507553993703}. Best is trial 0 with value: -1.45118023753422.
[I 2026-03-13 19:09:28,613] Trial 1 finished with value: -1.3262023998385273 and parameters: {'n_estimators': 737, 'learning_rate': 0.010725209743171996, 'max_depth': 12, 'subsample': 0.9162213204002109, 'colsample_bytree': 0.5274034664069657, 'min_child_weight': 2, 'gamma': 2.9324868872723725e-07, 'reg_alpha': 2.716051144654844e-06, 'reg_lambda': 0.00015777981883364995}. Best is trial 1 with value: -1.3262023998385273.
[I 2026-03-13 19:09:29,888] Trial 2 finished with value: -1.3304785651812543 and parameters: {'n_estimators': 489, 'learning_rate': 0.02692655251486473, 'max_dep

Best Optuna parameters: {'n_estimators': 238, 'learning_rate': 0.024893528510341163, 'max_depth': 10, 'subsample': 0.6648252398755385, 'colsample_bytree': 0.7170908662074246, 'min_child_weight': 4, 'gamma': 0.000237574238044691, 'reg_alpha': 1.846656718857583e-08, 'reg_lambda': 1.0553507893726066e-05}


Tuning completed in 435.21 seconds
Model:                   XGBoost (Optuna tuned)
MAE (log):               1.3669
RMSE (log):              1.9002
MAE (minutes):           1731.78 minutes
RMSE (minutes):          3663.05 minutes
R^2:                     0.5310
Training Time:           0.17 seconds


In [54]:
class OutageHurdleModel(BaseEstimator, RegressorMixin):
    def __init__(self, classifier, regressor):
        self.classifier = classifier
        self.regressor = regressor

    def _get_augmented_X(self, X, probs):
        # Adds the probability column as a new feature to the input matrix
        return np.column_stack([X, probs])

    def fit(self, X, y):
        # Stage 1: Train the gatekeeper
        y_binary = (y > 0).astype(int)
        print("Fitting Stage 1: Hurdle Classifier...")
        self.classifier.fit(X, y_binary)
        
        # Get probabilities for the training set to use as a feature for Stage 2
        probs = self.classifier.predict_proba(X)[:, 1]
        X_augmented = self._get_augmented_X(X, probs)
        
        # Stage 2: Train the estimator only on rows where duration > 0
        print("Fitting Stage 2: Duration Regressor (with Stage 1 context)...")
        mask = y_binary == 1
        self.regressor.fit(X_augmented[mask], y[mask])
        return self

    def predict(self, X):
        # 1. Get Stage 1 Probability
        probs = self.classifier.predict_proba(X)[:, 1]
        
        # 2. Augment X with that probability
        X_augmented = self._get_augmented_X(X, probs)
        
        # 3. Predict Raw Duration
        raw_preds = self.regressor.predict(X_augmented)
        
        # 4. Soft Hurdle: Probability * Predicted Duration
        return probs * raw_preds

In [55]:
# 1. Manually transform the features to numeric matrices
# This ensures Stage 1 and Stage 2 are looking at the same numeric data
X_train_proc = preprocessor_dense.fit_transform(X_train)
X_test_proc = preprocessor_dense.transform(X_test)

# 2. Initialize the sub-models with your best-found parameters
h_clf = RandomForestClassifier(n_estimators=200, max_depth=10, random_state=42)
h_reg = XGBRegressor(**optuna_xgb.named_steps['regressor'].get_params()) # Use the best XGBoost params found with Optuna

# 3. Initialize and fit the Hurdle Model
reaching_model = OutageHurdleModel(classifier=h_clf, regressor=h_reg)
reaching_model.fit(X_train_proc, y_train.values)

# 4. Generate Predictions
hurdle_preds = reaching_model.predict(X_test_proc)

Fitting Stage 1: Hurdle Classifier...
Fitting Stage 2: Duration Regressor (with Stage 1 context)...


In [56]:
y_test_minutes = np.expm1(y_test)
hurdle_preds_minutes = np.expm1(hurdle_preds)

# Calculate Final Metrics
OHM_metrics = {
    'Model': "Advanced Hurdle (RF + XGB Stacked)",
    'MAE (log)': mean_absolute_error(y_test, hurdle_preds),
    'RMSE (log)': root_mean_squared_error(y_test, hurdle_preds),
    'MAE (minutes)': mean_absolute_error(y_test_minutes, hurdle_preds_minutes),
    'RMSE (minutes)': root_mean_squared_error(y_test_minutes, hurdle_preds_minutes),
    'R^2': r2_score(y_test, hurdle_preds)
}

print("\n--- REAching MODEL PERFORMANCE ---")
for k, v in OHM_metrics.items():
    if isinstance(v, float):
        print(f"{k:15}: {v:.4f}")
    else:
        print(f"{k:15}: {v}")

results_log.append(OHM_metrics)


--- REAching MODEL PERFORMANCE ---
Model          : Advanced Hurdle (RF + XGB Stacked)
MAE (log)      : 1.3898
RMSE (log)     : 1.8705
MAE (minutes)  : 1786.1129
RMSE (minutes) : 3849.9978
R^2            : 0.5456


In [57]:
# Residuals for the OHM model
residual_df_ohm = pd.DataFrame({
    'Predicted Log Duration': hurdle_preds,
    'Residuals (log)': y_test - hurdle_preds,
    'Predicted Duration (min)': hurdle_preds_minutes,
    'Residuals (min)': y_test_minutes - hurdle_preds_minutes
})
ohm_fig = plot_log_and_reg_residuals(residual_df_ohm, "Advanced Hurdle Model")

# Export plot for website
ohm_fig.write_html('docs/assets/plots/advanced_hurdle_model_residuals.html', include_plotlyjs='cdn')

In [58]:
def objective_hurdle(trial):
    # Random Forest Classifier params 
    clf_params = {
        'n_estimators': trial.suggest_int('clf_n_estimators', 100, 800),
        'max_depth': trial.suggest_int('clf_max_depth', 3, 25),
        'min_samples_leaf': trial.suggest_int('clf_min_samples_leaf', 2, 20),
        'min_samples_split': trial.suggest_int('clf_min_samples_split', 2, 20),
        'max_features': trial.suggest_categorical('clf_max_features', ['sqrt', 'log2', 0.5]),
        'class_weight': trial.suggest_categorical('clf_class_weight', [None,'balanced']),
        'random_state': 42,
        'n_jobs': -1,
    }

    # XGBoost Regressor params 
    reg_params = {
        'n_estimators': trial.suggest_int('n_estimators', 100, 1000),
        'learning_rate': trial.suggest_float('learning_rate', 0.005, 0.2, log=True),
        'max_depth': trial.suggest_int('max_depth', 2, 12),
        'subsample': trial.suggest_float('subsample', 0.5, 1.0),
        'colsample_bytree': trial.suggest_float('colsample_bytree', 0.4, 1.0),
        'min_child_weight': trial.suggest_int('min_child_weight', 1, 10),
        'gamma': trial.suggest_float('gamma', 1e-8, 1.0, log=True),
        'reg_alpha': trial.suggest_float('reg_alpha', 1e-8, 1.0, log=True),
        'reg_lambda': trial.suggest_float('reg_lambda', 1e-8, 1.0, log=True),
        'tree_method': 'hist',
        'random_state': 42,
        'n_jobs': -1,
    }

    # Manual cross-validation 
    from sklearn.model_selection import KFold
    kf = KFold(n_splits=3, shuffle=True, random_state=42)
    fold_mae = []

    for train_idx, val_idx in kf.split(X_train_proc):
        X_tr, X_val = X_train_proc[train_idx], X_train_proc[val_idx]
        y_tr, y_val = y_train.values[train_idx], y_train.values[val_idx]

        model = OutageHurdleModel(
            classifier=RandomForestClassifier(**clf_params),
            regressor=XGBRegressor(**reg_params),
        )
        model.fit(X_tr, y_tr)
        preds = model.predict(X_val)
        fold_mae.append(mean_absolute_error(y_val, preds))
        
        # Prune unpromising trials after first fold
        trial.report(np.mean(fold_mae), step=len(fold_mae))
        if trial.should_prune():
            raise optuna.exceptions.TrialPruned()

    return np.mean(fold_mae)


study = optuna.create_study(
    direction='minimize',
    sampler=TPESampler(seed=42, n_startup_trials=20),
    pruner=MedianPruner(n_startup_trials=20, n_warmup_steps=1),
)
study.optimize(objective_hurdle, n_trials=1000, show_progress_bar=True)

print("Best MAE:", study.best_value)
print("Best params:", study.best_params)

[I 2026-03-13 19:16:40,486] A new study created in memory with name: no-name-e7fef154-0e40-4e3d-97b3-80755422c7b6


  0%|          | 0/1000 [00:00<?, ?it/s]

Fitting Stage 1: Hurdle Classifier...
Fitting Stage 2: Duration Regressor (with Stage 1 context)...
Fitting Stage 1: Hurdle Classifier...
Fitting Stage 2: Duration Regressor (with Stage 1 context)...
Fitting Stage 1: Hurdle Classifier...
Fitting Stage 2: Duration Regressor (with Stage 1 context)...
[I 2026-03-13 19:16:44,545] Trial 0 finished with value: 1.392185716115278 and parameters: {'clf_n_estimators': 362, 'clf_max_depth': 24, 'clf_min_samples_leaf': 15, 'clf_min_samples_split': 13, 'clf_max_features': 'sqrt', 'clf_class_weight': None, 'n_estimators': 737, 'learning_rate': 0.005394455304087533, 'max_depth': 12, 'subsample': 0.9162213204002109, 'colsample_bytree': 0.5274034664069657, 'min_child_weight': 2, 'gamma': 2.9324868872723725e-07, 'reg_alpha': 2.716051144654844e-06, 'reg_lambda': 0.00015777981883364995}. Best is trial 0 with value: 1.392185716115278.
Fitting Stage 1: Hurdle Classifier...
Fitting Stage 2: Duration Regressor (with Stage 1 context)...
Fitting Stage 1: Hurdle

In [59]:
best_OHM = study.best_params

best_clf = RandomForestClassifier(
    n_estimators=best_OHM['clf_n_estimators'],
    max_depth=best_OHM['clf_max_depth'],
    min_samples_leaf=best_OHM['clf_min_samples_leaf'],
    max_features=best_OHM['clf_max_features'],
    random_state=42,
)
best_reg = XGBRegressor(
    n_estimators=best_OHM['n_estimators'],
    learning_rate=best_OHM['learning_rate'],
    max_depth=best_OHM['max_depth'],
    subsample=best_OHM['subsample'],
    colsample_bytree=best_OHM['colsample_bytree'],
    min_child_weight=best_OHM['min_child_weight'],
    gamma=best_OHM['gamma'],
    reg_alpha=best_OHM['reg_alpha'],
    reg_lambda=best_OHM['reg_lambda'],
    random_state=42,
    objective='reg:squarederror',
)

reaching_model_tuned = OutageHurdleModel(classifier=best_clf, regressor=best_reg)
reaching_model_tuned.fit(X_train_proc, y_train.values)
tuned_hurdle_preds = reaching_model_tuned.predict(X_test_proc)

tuned_hurdle_preds_minutes = np.expm1(tuned_hurdle_preds)

# Calculate Final Metrics
tuned_OHM_metrics = {
    'Model': "Advanced Hurdle (Optuna tuned)",
    'MAE (log)': mean_absolute_error(y_test, tuned_hurdle_preds),
    'RMSE (log)': root_mean_squared_error(y_test, tuned_hurdle_preds),
    'MAE (minutes)': mean_absolute_error(y_test_minutes, tuned_hurdle_preds_minutes),
    'RMSE (minutes)': root_mean_squared_error(y_test_minutes, tuned_hurdle_preds_minutes),
    'R^2': r2_score(y_test, tuned_hurdle_preds)
}

print("\n--- REAching MODEL PERFORMANCE ---")
for k, v in tuned_OHM_metrics.items():
    if isinstance(v, float):
        print(f"{k:15}: {v:.4f}")
    else:
        print(f"{k:15}: {v}")

results_log.append(tuned_OHM_metrics)

Fitting Stage 1: Hurdle Classifier...
Fitting Stage 2: Duration Regressor (with Stage 1 context)...

--- REAching MODEL PERFORMANCE ---
Model          : Advanced Hurdle (Optuna tuned)
MAE (log)      : 1.3791
RMSE (log)     : 1.8828
MAE (minutes)  : 1794.9446
RMSE (minutes) : 3793.8704
R^2            : 0.5396


In [60]:
# Residuals for tuned OHM model
residual_df_tuned_ohm = pd.DataFrame({
    'Predicted Log Duration': tuned_hurdle_preds,
    'Residuals (log)': y_test - tuned_hurdle_preds,
    'Predicted Duration (min)': tuned_hurdle_preds_minutes,
    'Residuals (min)': y_test_minutes - tuned_hurdle_preds_minutes
})
tuned_ohm_fig = plot_log_and_reg_residuals(residual_df_tuned_ohm, "Advanced Hurdle Model (Optuna Tuned)")
tuned_ohm_fig.write_html('docs/assets/plots/advanced_hurdle_model_residuals_tuned.html', include_plotlyjs='cdn')

In [70]:
results_df = pd.DataFrame(results_log)
results_df['Model'] = results_df['Model'].replace({
    'HistGradientBoostingRegressor (tuned)': 'HGB Regressor (tuned)'
})
results_df

,Model,MAE (log),RMSE (log),MAE (minutes),RMSE (minutes),R^2
0,Linear Regression (Baseline),1.46,1.93,1812.06,4378.00,0.52
1,Ridge Regression (tuned),1.48,1.91,1802.69,3925.18,0.52
2,Random Forest Regressor (tuned),1.34,1.83,1739.02,3719.70,0.57
3,XGBoost Regressor (tuned),1.37,1.90,1755.67,3729.29,0.53
4,HGB Regressor (tuned),1.34,1.86,1691.86,3557.68,0.55
5,XGBoost (Optuna tuned),1.37,1.90,1731.78,3663.05,0.53
6,Advanced Hurdle (RF + XGB Stacked),1.39,1.87,1786.11,3850.00,0.55
7,Advanced Hurdle (Optuna tuned),1.38,1.88,1794.94,3793.87,0.54


In [126]:
# To markdown for website
print(results_df.to_markdown(index=False))

| Model                              |   MAE (log) |   RMSE (log) |   MAE (minutes) |   RMSE (minutes) |      R^2 |
|:-----------------------------------|------------:|-------------:|----------------:|-----------------:|---------:|
| Linear Regression (Baseline)       |     1.46448 |      1.92543 |         1812.06 |          4378    | 0.518461 |
| Ridge Regression (tuned)           |     1.47886 |      1.91423 |         1802.69 |          3925.18 | 0.524049 |
| Random Forest Regressor (tuned)    |     1.33684 |      1.82861 |         1739.02 |          3719.7  | 0.565669 |
| XGBoost Regressor (tuned)          |     1.37451 |      1.89939 |         1755.67 |          3729.29 | 0.531397 |
| HGB Regressor (tuned)              |     1.34207 |      1.86236 |         1691.86 |          3557.68 | 0.549491 |
| XGBoost (Optuna tuned)             |     1.36692 |      1.90019 |         1731.78 |          3663.05 | 0.531003 |
| Advanced Hurdle (RF + XGB Stacked) |     1.38978 |      1.87048 |     

In [139]:
baseline = results_df[results_df['Model'] == 'Linear Regression (Baseline)'].iloc[0]
best = results_df[results_df['Model'] == 'Random Forest Regressor (tuned)'].iloc[0]

metrics = ['MAE (log)', 'RMSE (log)', 'MAE (minutes)', 'RMSE (minutes)', 'R^2']

comparison_df = pd.DataFrame({
    'Metric': metrics,
    'Linear Regression (Baseline)': [baseline[m] for m in metrics],
    'Random Forest Regressor (tuned)': [best[m] for m in metrics],
})

# For R^2 higher is better, for everything else lower is better
def pct_improvement(metric, base_val, best_val):
    if metric == 'R^2':
        return (best_val - base_val) / abs(base_val) * 100
    else:
        return (base_val - best_val) / abs(base_val) * 100
    
comparison_df['Absolute Improvement'] = comparison_df.apply(
    lambda row: np.abs(row['Linear Regression (Baseline)'] - row['Random Forest Regressor (tuned)']),
    axis=1
)
comparison_df['% Improvement'] = comparison_df.apply(
    lambda row: pct_improvement(row['Metric'], row['Linear Regression (Baseline)'], row['Random Forest Regressor (tuned)']),
    axis=1
)

comparison_df['% Improvement'] = comparison_df['% Improvement'].map('{:+.1f}%'.format)

comparison_df

,Metric,Linear Regression (Baseline),Random Forest Regressor (tuned),Absolute Improvement,% Improvement
0,MAE (log),1.46,1.34,0.13,+8.7%
1,RMSE (log),1.93,1.83,0.10,+5.0%
2,MAE (minutes),1812.06,1739.02,73.03,+4.0%
3,RMSE (minutes),4378.00,3719.70,658.29,+15.0%
4,R^2,0.52,0.57,0.05,+9.1%


In [140]:
print(comparison_df.to_markdown(index=False))

| Metric         |   Linear Regression (Baseline) |   Random Forest Regressor (tuned) |   Absolute Improvement | % Improvement   |
|:---------------|-------------------------------:|----------------------------------:|-----------------------:|:----------------|
| MAE (log)      |                       1.46448  |                          1.33684  |              0.127637  | +8.7%           |
| RMSE (log)     |                       1.92543  |                          1.82861  |              0.0968146 | +5.0%           |
| MAE (minutes)  |                    1812.06     |                       1739.02     |             73.0343    | +4.0%           |
| RMSE (minutes) |                    4378        |                       3719.7      |            658.295     | +15.0%          |
| R^2            |                       0.518461 |                          0.565669 |              0.0472081 | +9.1%           |


In [63]:
metrics = ['MAE (log)', 'RMSE (log)', 'MAE (minutes)', 'RMSE (minutes)', 'R^2']
ascending = {'MAE (log)': False, 'RMSE (log)': False, 'MAE (minutes)': False, 'RMSE (minutes)': False, 'R^2': True}

fig = go.Figure()

for i, metric in enumerate(metrics):
    ordered = results_df.sort_values(metric, ascending=ascending[metric])
    fig.add_trace(go.Bar(
        y=ordered['Model'],
        x=ordered[metric],
        name=metric,
        visible=(i == 0),
        text=ordered[metric].apply(lambda x: f'{x:.4f}'),
        textposition='outside',
        marker_color=ordered[metric],
        marker_colorscale='Teal',
        orientation='h'
    ))

fig.update_layout(
    title=f'<b>Model Performance Comparison - {metrics[0]}</b>',
    showlegend=False,
    template='plotly_white',
    updatemenus=[dict(
        buttons=[
            dict(
                label=metric,
                method='update',
                args=[
                    {'visible': [j == i for j in range(len(metrics))]},
                    {'title': f'<b>Model Performance Comparison - {metric}</b>'},
                ]
            )
            for i, metric in enumerate(metrics)
        ],
        direction='down',
        x=1.0,
        xanchor='right',
        y=1.15,
        yanchor='top',
        showactive=True,
    )]
)

fig.show()

# Export plot for website
fig.write_html('docs/assets/plots/model_performance_comparison.html', include_plotlyjs='cdn')

## Step 8: Fairness Analysis

Our approach for the model's fairness analysis will be comparing its performance in urban vs rural states. We will quantify "urban states" as those where the POPPCT_URBAN (Percentage of the total population of the U.S. state represented by the urban population (in %)) is above the median, with "rural states" being below. 

**Null Hypothesis:** Our model is fair with respect to urban states and rural states, and any observed difference (in RMSE) is due to random chance.
**Alternative Hypothesis:** Our model is unfair with respect to urban states and rural states. The RMSE differs significantly between urban and rural states.     

We will be using the OutageHurdleModel as our selected best model. 

In [142]:
# Selecting OutageHurdleModel as best model, but can use any to apply methodology
selected_model = reaching_model_tuned

# Preprocess X_test before passing to hurdle model
X_test_proc  = preprocessor_dense.transform(X_test)
y_pred_log = selected_model.predict(X_test_proc)

y_test_minutes = np.expm1(y_test)
y_pred_minutes = np.expm1(y_pred_log)

In [65]:
# Pull POPPCT_URBAN from df_engineered using original index
poppct_urban_test = df_engineered.loc[X_test.index, 'POPPCT_URBAN'].to_numpy()
urban_threshold = np.median(poppct_urban_test)
is_urban = (poppct_urban_test >= urban_threshold)
is_rural = ~is_urban

print(f"Median POPPCT_URBAN in test set: {urban_threshold:.2f}%")
print(f"Urban group size: {is_urban.sum()}")
print(f"Rural group size: {is_rural.sum()}")

Median POPPCT_URBAN in test set: 84.05%
Urban group size: 162
Rural group size: 128


In [66]:
# Minutes scale
rmse_urban = root_mean_squared_error(y_test_minutes[is_urban], y_pred_minutes[is_urban])
rmse_rural = root_mean_squared_error(y_test_minutes[is_rural], y_pred_minutes[is_rural])

observed_diff = np.abs(rmse_rural - rmse_urban)

print(f"RMSE (urban): {rmse_urban:.0f} minutes")
print(f"RMSE (rural): {rmse_rural:.0f} minutes")
print(f"RMSE Absolute Observed Difference: {observed_diff:.0f} minutes ({observed_diff/60:.1f} hours)")

RMSE (urban): 4024 minutes
RMSE (rural): 3481 minutes
RMSE Absolute Observed Difference: 543 minutes (9.1 hours)


In [67]:
mae_urban = mean_absolute_error(y_test_minutes[is_urban], y_pred_minutes[is_urban])
mae_rural = mean_absolute_error(y_test_minutes[is_rural], y_pred_minutes[is_rural])

mae_diff = np.abs(mae_rural - mae_urban)

print(f"MAE (Urban): {mae_urban:.0f} minutes")
print(f"MAE (Rural): {mae_rural:.0f} minutes")
print(f"MAE Absolute Observed Difference: {mae_diff:.0f} minutes")

MAE (Urban): 1811 minutes
MAE (Rural): 1774 minutes
MAE Absolute Observed Difference: 37 minutes


In [68]:
np.random.seed(42)
fairness_diffs = []
for _ in range(1000):
    shuffled_urban = np.random.permutation(is_urban)
    shuffled_rural = ~shuffled_urban # For readability

    rmse_urban_shuffled = root_mean_squared_error(y_test_minutes[shuffled_urban], y_pred_minutes[shuffled_urban])
    rmse_rural_shuffled = root_mean_squared_error(y_test_minutes[shuffled_rural], y_pred_minutes[shuffled_rural])
    fairness_diffs.append(np.abs(rmse_rural_shuffled - rmse_urban_shuffled))

fairness_diffs = np.array(fairness_diffs)
p_value = np.mean(fairness_diffs >= observed_diff)

print(f"P-value: {p_value:.4f}")
print(f"Conclusion: {'Reject H0: evidence of unfairness' if p_value < 0.05 else 'Fail to reject H0: no significant evidence of unfairness'}")

P-value: 0.5080
Conclusion: Fail to reject H0: no significant evidence of unfairness


In [69]:
fig = px.histogram(
    x=fairness_diffs, nbins=50,
    title='<b>Fairness Test: Permutation Test of Absolute RMSE Differences (Urban - Rural)</b>',
    labels={'x': 'Absolute RMSE Differences (minutes)'},
)

fig.add_vline(
    x=observed_diff,
    line_dash='dash',
    line_color='red',
    line_width=2,
    annotation_text=f'Observed Diff: {observed_diff:.0f} min',
    annotation_position='top right',
    annotation=dict(font_size=12, font_color='red')
)

fig.update_layout(template='plotly_white', bargap=0.03)
fig.show()

# Export plot for website
fig.write_html('docs/assets/plots/fairness_permutation_test.html', include_plotlyjs='cdn')

Our observed test statistic was the difference in RMSE between rural and urban states, where we performed a permutation test (n=1000) that yielded a p-value of 0.5080. As such, we do not have sufficient evidence to conclude that the model is less fair for rural states.

Because the model was trained on log-transformed outage duration but evaluated here on the original minutes scale, this fairness result reflects real-world prediction error in outage duration.